# matvis: Comprehensive Documentation of the Visibility Simulator

**Author:** Auto-generated documentation notebook  
**Date:** April 13, 2026  
**Package Location:** `/lustre/aoc/projects/hera/rchandra/miniconda3/envs/myenv_validation_1/lib/python3.10/site-packages/matvis/`

---

This notebook provides a complete, step-by-step description of how `matvis` implements the Radio Interferometer Measurement Equation (RIME) to simulate visibilities. Every numerical operation, normalization choice, and algorithmic decision is documented with the corresponding mathematics.

### Master Index Legend (used throughout this notebook)

The following table defines **every** index, subscript, and superscript that appears in the equations below. Refer back to this table whenever a symbol is unclear.

| Symbol | What it labels | Meaning | Range / Typical Values |
|--------|---------------|---------|------------------------|
| $p, q$ | **Antenna** indices | Label two specific physical antennas in the array that form an interferometric baseline | $p, q \in \{1, 2, \ldots, N_{\text{ant}}\}$; e.g. for HERA-350, $N_{\text{ant}} = 350$ |
| $s$ | **Source** index | Labels a single point source from the sky catalog | $s \in \{1, 2, \ldots, N_{\text{src}}\}$ |
| $k$ | **Frequency channel** index | Labels a discrete frequency channel in the observation band | $k \in \{1, 2, \ldots, N_{\text{freq}}\}$ |
| $t$ | **Time step** index | Labels a discrete time integration (an LST snapshot) | $t \in \{1, 2, \ldots, N_{\text{time}}\}$ |
| $\alpha, \beta$ | **Antenna feed** polarization indices | Label which of the two physical orthogonal dipole feeds on an antenna is being used. Each antenna has two feeds: one oriented East–West (called $X$ or feed 1) and one oriented North–South (called $Y$ or feed 2). $\alpha$ refers to the feed on antenna $p$; $\beta$ refers to the feed on antenna $q$. | $\alpha, \beta \in \{X, Y\}$ (2 values) |
| $\gamma, \delta$ | **Sky polarization** indices | Label the two orthogonal electric-field directions on the celestial sphere in a spherical coordinate system: $\hat{\theta}$ (pointing along increasing zenith angle) and $\hat{\phi}$ (pointing along increasing azimuth). These are properties of the incoming electromagnetic wave, not the antenna. | $\gamma, \delta \in \{\hat{\theta}, \hat{\phi}\}$ (2 values) |
| $i$ | **Spatial component** index | Labels one of the three Cartesian coordinate axes (East, North, Up) in the topocentric frame | $i \in \{E, N, U\}$ |
| $\nu$ | **Frequency** (continuous) | Observing frequency | Units: Hz |
| $\nu_k$ | **Frequency** of channel $k$ | Discrete frequency of the $k$-th channel | Units: Hz |

**Critical distinction between $(\alpha, \beta)$ and $(\gamma, \delta)$:**
- $\alpha, \beta$ live on the **antenna hardware** — they select which physical dipole feed's voltage output we are reading.
- $\gamma, \delta$ live on the **sky** — they describe the direction of the incoming electric field oscillation on the celestial sphere.
- The Jones matrix $J_p^{\alpha\gamma}$ **connects** these two worlds: it tells us how much voltage feed $\alpha$ produces in response to electric field component $\gamma$.

## Table of Contents

1. [Overview & the Analytic RIME](#1-overview)
2. [Coordinate System & Source Position Computation](#2-coordinates)
3. [Beam Evaluation & Interpolation](#3-beam)
4. [Beam Normalization: Peak vs. Area](#4-beam-norm)
5. [Source Coherency Matrix Construction](#5-coherency)
6. [Fringe (Phase) Computation](#6-fringe)
7. [The Core Matrix Multiplication (Visibility Formation)](#7-core-multiplication)
8. [Polarization Handling & Feed Mapping](#8-polarization)
9. [Frequency & Time Looping Structure](#9-loops)
10. [GPU vs CPU Implementation Differences](#10-gpu-cpu)
11. [Summary of All Numerical Choices](#11-summary)

---
## 1. Overview & the Analytic RIME <a id='1-overview'></a>

### 1.1 The Analytic (Continuous) RIME

The full-sky Radio Interferometer Measurement Equation (RIME) for baseline $(p, q)$ at frequency $\nu$ is:

$$
\mathbf{V}_{pq}(\nu) = \int_{4\pi} \mathbf{J}_p(\hat{\mathbf{s}}, \nu) \; \mathbf{C}(\hat{\mathbf{s}}, \nu) \; \mathbf{J}_q^H(\hat{\mathbf{s}}, \nu) \; e^{-2\pi i \nu \, \mathbf{b}_{pq} \cdot \hat{\mathbf{s}} / c} \; d\Omega
$$

> **Index legend for this equation:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $p$ | Index of the first antenna in the baseline pair. This is a physical antenna element in the array (e.g., antenna #42 in HERA). |
> | $q$ | Index of the second antenna in the baseline pair. Together, $(p, q)$ define one interferometric baseline. |
> | $\nu$ | Observing frequency in Hz. |
> | $\hat{\mathbf{s}}$ | Unit direction vector pointing toward a location on the celestial sphere. This is the integration variable — we integrate over **all** sky directions. |
> | $\mathbf{V}_{pq}(\nu)$ | The $2 \times 2$ **visibility matrix** for baseline $pq$ at frequency $\nu$. Its 4 elements correspond to the 4 possible feed-pair correlations: $XX, XY, YX, YY$. Each element is a complex number. Units: Jy. |
> | $\mathbf{J}_p(\hat{\mathbf{s}}, \nu)$ | The $2 \times 2$ **Jones matrix** of antenna $p$. Rows indexed by feed $\alpha \in \{X, Y\}$, columns by sky polarization $\gamma \in \{\hat{\theta}, \hat{\phi}\}$. It describes how the antenna beam converts incoming sky electric fields into feed voltages. |
> | $\mathbf{C}(\hat{\mathbf{s}}, \nu)$ | The $2 \times 2$ **sky coherency matrix** in direction $\hat{\mathbf{s}}$ at frequency $\nu$. It encodes the intensity and polarization state of the sky emission. Rows/columns indexed by sky polarizations $(\gamma, \delta)$. Units for extended emission: Jy/sr. |
> | $\mathbf{J}_q^H$ | The **conjugate transpose** (Hermitian adjoint) of the Jones matrix of antenna $q$. The superscript $H$ means: complex-conjugate every element, then transpose the matrix. This arises because the visibility is a correlation $\langle v_p \, v_q^* \rangle$, where the second antenna's voltage is conjugated. |
> | $\mathbf{b}_{pq}$ | The **baseline vector**: $\mathbf{b}_{pq} = \mathbf{r}_p - \mathbf{r}_q$, where $\mathbf{r}_p$ and $\mathbf{r}_q$ are the 3D position vectors of antennas $p$ and $q$ in the topocentric ENU frame. Units: meters. |
> | $c$ | Speed of light, $2.998 \times 10^8$ m/s. |
> | $d\Omega$ | Differential solid angle element on the celestial sphere (steradians). $d\Omega = \sin\theta\,d\theta\,d\phi$. The full sphere integrates to $\int_{4\pi} d\Omega = 4\pi$. |
> | $e^{-2\pi i \nu \mathbf{b}_{pq} \cdot \hat{\mathbf{s}} / c}$ | The **fringe** (geometric phase). Encodes the phase delay between antennas $p$ and $q$ for a wave arriving from direction $\hat{\mathbf{s}}$. |

**Physical meaning:** This integral says: the measured cross-correlation (visibility) between antennas $p$ and $q$ equals the sky brightness in every direction, weighted by (a) how sensitive each antenna's beam is in that direction (Jones matrices), (b) the polarization state of the sky (coherency), and (c) an oscillating fringe pattern that depends on the antenna separation. The integral sums these weighted contributions over the entire sky.

### 1.2 The Discrete RIME (as implemented by matvis)

`matvis` discretizes the sky into a catalogue of $N_{\text{src}}$ point sources. Each source $s$ sits at a known direction $\hat{\mathbf{s}}_s$ on the sky and has a known flux density. The continuous integral becomes a **finite summation**:

$$
\mathbf{V}_{pq}(\nu) = \sum_{s=1}^{N_{\text{src}}} \mathbf{J}_p(\hat{\mathbf{s}}_s, \nu) \; \mathbf{C}_s(\nu) \; \mathbf{J}_q^H(\hat{\mathbf{s}}_s, \nu) \; e^{-2\pi i \nu \, \mathbf{b}_{pq} \cdot \hat{\mathbf{s}}_s / c}
$$

> **Index legend for this equation:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $s$ | Source index running over all point sources in the sky catalog, from $1$ to $N_{\text{src}}$. |
> | $N_{\text{src}}$ | Total number of discrete point sources in the catalog. |
> | $\hat{\mathbf{s}}_s$ | Unit direction vector toward the $s$-th source on the sky. |
> | $\mathbf{C}_s(\nu)$ | The $2 \times 2$ coherency matrix **of source $s$ specifically** (not per steradian). For an unpolarized source: $\mathbf{C}_s = I_s \, \mathbb{I}_2$. Units: **Jy** (total source flux, not Jy/sr). |
> | All other symbols | Same as Section 1.1 above. |

**Key point:** There is **no $d\Omega$ factor** because the sources are point sources. Their flux densities (in Jy) already represent the total brightness of each source. This is the fundamental difference from simulating continuous diffuse emission, where you would need to multiply by a pixel solid angle $\Delta\Omega$.

### 1.3 matvis's Reformulation as a Matrix Product

The key computational insight of `matvis` is to reformulate the RIME as **matrix multiplications** rather than looping over baselines one by one.

**What is a "single polarization feed pair $(\alpha, \beta)$"?** This refers to choosing one specific feed on antenna $p$ (say the $X$-feed, so $\alpha = X$) and one specific feed on antenna $q$ (say the $Y$-feed, so $\beta = Y$). The resulting visibility $V_{pq}^{XY}$ is the cross-correlation of the $X$-feed voltage of antenna $p$ with the $Y$-feed voltage of antenna $q$. In a dual-polarization system, there are four such pairs: $(X,X)$, $(X,Y)$, $(Y,X)$, $(Y,Y)$.

For one such feed pair $(\alpha, \beta)$ the visibility is:

$$
V_{pq}^{\alpha\beta}(\nu) = \sum_{s=1}^{N_{\text{src}}} \left[ \sum_{\gamma} J_p^{\alpha\gamma}(\hat{\mathbf{s}}_s, \nu) \sqrt{C_s^{\gamma\delta}(\nu)} \right] \; e^{-2\pi i \nu \, \mathbf{b}_{pq} \cdot \hat{\mathbf{s}}_s / c} \; \left[ \sum_{\delta} J_q^{\beta\delta *}(\hat{\mathbf{s}}_s, \nu) \sqrt{C_s^{\gamma\delta *}(\nu)} \right]
$$

> **Index legend for this equation:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $V_{pq}^{\alpha\beta}$ | A single **scalar** (complex number): the visibility for the specific feed pair $(\alpha, \beta)$. It is one of the 4 elements of the $2\times 2$ visibility matrix $\mathbf{V}_{pq}$. |
> | $\alpha$ | Feed polarization of antenna $p$: $\alpha \in \{X, Y\}$. Physically, this selects which dipole feed on antenna $p$ we read. For HERA, $X$ is typically the East–West dipole, $Y$ the North–South dipole. |
> | $\beta$ | Feed polarization of antenna $q$: $\beta \in \{X, Y\}$. Same meaning as $\alpha$, but for the second antenna. |
> | $\gamma$ | Sky polarization index summed over: $\gamma \in \{\hat{\theta}, \hat{\phi}\}$. This sums over the two orthogonal electric-field directions on the sky. |
> | $\delta$ | Second sky polarization index (for the coherency matrix column). |
> | $J_p^{\alpha\gamma}$ | One scalar element of the Jones matrix: the complex voltage response of feed $\alpha$ on antenna $p$ to an incoming wave with sky polarization $\gamma$, from the direction of source $s$. |
> | $J_q^{\beta\delta *}$ | The complex conjugate of the Jones element for feed $\beta$ on antenna $q$ responding to sky polarization $\delta$. The $*$ comes from the $\mathbf{J}_q^H$ in the RIME. |
> | $C_s^{\gamma\delta}$ | Element $(\gamma, \delta)$ of the coherency matrix of source $s$. |

More precisely, `matvis` constructs **per-antenna** vectors and then takes outer products. For the **Stokes I only** (unpolarized) case, define:

$$
A_p^{\alpha}(s, \nu) = J_p^{\alpha\gamma}(\hat{\mathbf{s}}_s, \nu) \cdot \sqrt{I_s(\nu)} \cdot e^{-2\pi i \nu \, \hat{\mathbf{s}}_s \cdot \mathbf{r}_p / c}
$$

> **Index legend for this equation:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $A_p^{\alpha}(s, \nu)$ | A single complex number: the "per-antenna measurement element" for antenna $p$, feed $\alpha$, from source $s$, at frequency $\nu$. |
> | $J_p^{\alpha\gamma}(\hat{\mathbf{s}}_s, \nu)$ | Beam Jones element of antenna $p$, feed $\alpha$, responding to sky polarization $\gamma$, in the direction of source $s$. In the Stokes I scalar-beam case, $\gamma$ is implicit (only the dominant diagonal term is used). |
> | $\sqrt{I_s(\nu)}$ | Positive real square root of the flux density (Jy) of source $s$ at frequency $\nu$. Units: $\sqrt{\text{Jy}}$. |
> | $\mathbf{r}_p$ | 3D position vector of antenna $p$ in the topocentric ENU frame (meters). |
> | $e^{-2\pi i \nu \hat{\mathbf{s}}_s \cdot \mathbf{r}_p / c}$ | Per-antenna geometric phase phasor for antenna $p$ toward source $s$. |

Then the visibility is computed as:

$$
V_{pq}^{\alpha\beta}(\nu) = \sum_{s} A_p^{\alpha}(s, \nu) \; A_q^{\beta *}(s, \nu) = \mathbf{a}_p^\alpha \cdot (\mathbf{a}_q^\beta)^H
$$

> **Index legend for this equation:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $A_q^{\beta *}$ | Complex conjugate of $A_q^{\beta}$. The conjugation implements the $\mathbf{J}_q^H$ from the RIME. |
> | $\mathbf{a}_p^\alpha$ | A row vector of length $N_{\text{src}}$, containing $A_p^\alpha(s, \nu)$ for all sources $s$. This is row $p$ of the matrix $\mathbf{A}^\alpha$. |
> | $(\mathbf{a}_q^\beta)^H$ | Conjugate-transpose of the vector for antenna $q$, feed $\beta$: a column vector of length $N_{\text{src}}$. |

This is computed as the matrix product $\mathbf{V} = \mathbf{A} \mathbf{A}^H$ where $\mathbf{A}$ has shape $(N_{\text{ant}}, N_{\text{src}})$.

**This is the fundamental computational strategy of matvis:** It avoids explicit loops over $N_{\text{bl}} = N_{\text{ant}}(N_{\text{ant}}-1)/2$ baselines. Instead, it builds the $(N_{\text{ant}} \times N_{\text{src}})$ matrix $\mathbf{A}$ and calls a single highly-optimized BLAS matrix multiplication $\mathbf{A} \mathbf{A}^H$, which computes all $N_{\text{ant}}^2$ baseline visibilities at once in $O(N_{\text{ant}}^2 \times N_{\text{src}})$ time.

In [ ]:
# Let's inspect the matvis package structure
import importlib
import inspect
import matvis

print("matvis version:", matvis.__version__)
print("\nPackage location:", matvis.__file__)
print("\nPublic API:")
for name in sorted(dir(matvis)):
    if not name.startswith('_'):
        obj = getattr(matvis, name)
        print(f"  {name}: {type(obj).__name__}")

In [ ]:
# Explore the source tree
import os
matvis_path = os.path.dirname(matvis.__file__)
print(f"matvis root: {matvis_path}\n")

for root, dirs, files in os.walk(matvis_path):
    # Skip __pycache__
    dirs[:] = [d for d in dirs if d != '__pycache__']
    level = root.replace(matvis_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in sorted(files):
        if file.endswith('.py') or file.endswith('.pyx'):
            filepath = os.path.join(root, file)
            size = os.path.getsize(filepath)
            print(f"{subindent}{file}  ({size} bytes)")

---
## 2. Coordinate System & Source Position Computation <a id='2-coordinates'></a>

### 2.1 Overview

The first step in the simulation pipeline is to compute where each source appears on the sky **relative to the array** at a given time. Source catalogs provide positions in the **equatorial frame** (Right Ascension, Declination), which is fixed to the distant stars. But the antenna beam pattern is bolted to the **ground** (topocentric frame). As the Earth rotates, the apparent position of each source in the topocentric frame changes — this is the basis of Earth-rotation aperture synthesis.

### 2.2 Equatorial to Topocentric Transformation

Given a source at Right Ascension $\alpha_{\text{ra}}$ and Declination $\delta$ observed at Local Sidereal Time (LST) from a site at geographic latitude $\phi_{\text{lat}}$:

**Hour Angle:**
$$h = \text{LST} - \alpha_{\text{ra}}$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $h$ | **Hour angle** of the source: how far (in angle) it has rotated past the local meridian. $h = 0$ means the source is transiting (at its highest point). $h > 0$ means the source is west of the meridian (it has already transited). Units: radians. |
> | $\text{LST}$ | **Local Sidereal Time**: the right ascension currently crossing the local meridian. Advances by $2\pi$ radians in one sidereal day (~23h 56m). |
> | $\alpha_{\text{ra}}$ | **Right Ascension** of the source: its "celestial longitude" in the equatorial coordinate system (radians). |
> | $\delta$ | **Declination** of the source: its "celestial latitude" (radians). $\delta = 0$ is the celestial equator, $\delta = +\pi/2$ is the north celestial pole. |
> | $\phi_{\text{lat}}$ | **Geographic latitude** of the observatory. For HERA: $\phi_{\text{lat}} \approx -30.7°$. |

**Topocentric direction cosines** (ENU frame):

The source unit vector in the topocentric frame is computed from equatorial coordinates via direct trigonometric transforms to obtain the direction cosines $(l, m, n)$:

$$l = \cos(\delta)\sin(h)$$
$$m = \sin(\delta)\cos(\phi_{\text{lat}}) - \cos(\delta)\cos(h)\sin(\phi_{\text{lat}})$$
$$n = \sin(\delta)\sin(\phi_{\text{lat}}) + \cos(\delta)\cos(h)\cos(\phi_{\text{lat}})$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $l$ | **East** direction cosine: the projection of the source unit vector $\hat{\mathbf{s}}$ onto the local East axis. |
> | $m$ | **North** direction cosine: the projection onto the local North axis. |
> | $n$ | **Up** (zenith) direction cosine: the projection onto the local vertical. $n = \cos(\theta_{\text{zenith}}) = \sin(\text{altitude})$. A source at the zenith has $n=1$; a source on the horizon has $n=0$. |
> | $(l, m, n)$ | Together these form a unit vector on the sky: $l^2 + m^2 + n^2 = 1$. This is the ENU (East-North-Up) representation. |
> | $\delta$ | Declination of the source (same as above). |
> | $h$ | Hour angle (same as above). |
> | $\phi_{\text{lat}}$ | Observatory latitude (same as above). |

**Physical picture:** Stand at the center of the array and look up. The $l$-axis points East, the $m$-axis points North, and the $n$-axis points to the zenith. A source directly overhead has $(l, m, n) = (0, 0, 1)$. A source on the eastern horizon has $(l, m, n) = (1, 0, 0)$.

### 2.3 Horizon Cut

Sources below the horizon are excluded from the sum:

$$\text{Source } s \text{ is above horizon if } n_s > 0$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $n_s$ | The Up (zenith) direction cosine of source $s$ at the current time step $t$. This changes with time as the Earth rotates. |

This is implemented as a boolean mask applied to the source array. Physically, the ground blocks radiation from sources below the horizon, so they cannot contribute to the measured visibility. Note that the set of above-horizon sources changes at each time step.

### 2.4 Conversion to Spherical Angles for Beam Evaluation

Beam models (both analytic and tabulated) expect input in spherical coordinates rather than direction cosines. `matvis` computes the **zenith angle** $\theta_s$ and **azimuth** $\phi_{\text{az},s}$ for each above-horizon source $s$:

$$\theta_s = \arccos(n_s)$$

$$\phi_{\text{az},s} = \arctan2(l_s, m_s)$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $\theta_s$ | **Zenith angle** of source $s$: angle between the source direction and the zenith. $\theta = 0$ at zenith, $\theta = \pi/2$ at the horizon. |
> | $\phi_{\text{az},s}$ | **Azimuth** of source $s$: angle measured in the horizontal plane. Convention: $\phi_{\text{az}} = 0$ is North, $\pi/2$ is East, $\pi$ is South, $3\pi/2$ is West. |
> | $l_s, m_s, n_s$ | The ENU direction cosines of source $s$ (defined above). |

**Numerical convention note:** The use of `arctan2(l, m)` (i.e., `arctan2(East, North)`) defines azimuth measured **East from North**. This is the standard astronomical azimuth convention. The `arctan2` function (as opposed to `arctan`) correctly handles all four quadrants.

### 2.5 The `crd_eq` and `eq2tops` Approach

In practice, `matvis` does not use the trigonometric formulas above directly. Instead it takes as input:

- **`crd_eq`**: Equatorial unit vectors of shape $(3, N_{\text{src}})$ — these are the $(x, y, z)$ components of each source direction in the equatorial Cartesian frame:
  $$\hat{\mathbf{s}}_{\text{eq},s} = (\cos\delta_s \cos\alpha_{\text{ra},s}, \; \cos\delta_s \sin\alpha_{\text{ra},s}, \; \sin\delta_s)$$

  > **Index legend:** $s$ is the source index. The three components $(x, y, z)_{\text{eq}}$ are the standard conversion from spherical (RA, Dec) to Cartesian on the unit sphere. $x_{\text{eq}}$ points toward the vernal equinox, $z_{\text{eq}}$ toward the north celestial pole.

- **`eq2tops`**: A rotation matrix of shape $(N_{\text{time}}, 3, 3)$ that rotates from equatorial to topocentric (ENU) coordinates at each time step $t$. This single $3 \times 3$ matrix encodes both the observatory latitude and the current LST.

The topocentric coordinates at time $t$ are then:

$$\hat{\mathbf{s}}_{\text{top}}(t) = \mathbf{R}_{\text{eq2top}}(t) \cdot \hat{\mathbf{s}}_{\text{eq}}$$

> **Index legend:**
>
> | Symbol | Shape | Meaning |
> |--------|-------|---------|
> | $\hat{\mathbf{s}}_{\text{eq}}$ | $(3, N_{\text{src}})$ | Source unit vectors in the equatorial frame. |
> | $\mathbf{R}_{\text{eq2top}}(t)$ | $(3, 3)$ | Rotation matrix at time $t$. Proper orthogonal: $\mathbf{R}^T \mathbf{R} = \mathbb{I}$. |
> | $\hat{\mathbf{s}}_{\text{top}}(t)$ | $(3, N_{\text{src}})$ | Source directions in the topocentric ENU frame at time $t$. Row 0 = East ($l$), Row 1 = North ($m$), Row 2 = Up ($n$). |

The third row of $\hat{\mathbf{s}}_{\text{top}}$ gives the $n$ direction cosine, which is used for the horizon cut ($n > 0$).

In [ ]:
# Let's look at the core simulate function signature and docstring
# to confirm the input parameters
try:
    from matvis import simulate
    print(inspect.signature(simulate))
    print("\n--- Docstring (first 3000 chars) ---")
    print(inspect.getdoc(simulate)[:3000])
except Exception as e:
    # Try alternative entry points
    print(f"Direct import failed: {e}")
    try:
        from matvis import simulate_vis
        print(inspect.signature(simulate_vis))
        print("\n--- Docstring (first 3000 chars) ---")
        print(inspect.getdoc(simulate_vis)[:3000])
    except:
        # Explore what's available
        print("Available in matvis:", [x for x in dir(matvis) if 'sim' in x.lower()])

In [ ]:
# Read the core simulation source code
import glob

matvis_path = os.path.dirname(matvis.__file__)

# Find the main simulation files
core_files = []
for pattern in ['**/core*.py', '**/simulate*.py', '**/cpu*.py', '**/engine*.py', '**/wrapper*.py']:
    core_files.extend(glob.glob(os.path.join(matvis_path, pattern), recursive=True))

for f in sorted(set(core_files)):
    print(f"\n{'='*80}")
    print(f"FILE: {f}")
    print(f"{'='*80}")
    with open(f, 'r') as fh:
        content = fh.read()
        # Print first 5000 chars to understand structure
        print(content[:5000])
        if len(content) > 5000:
            print(f"\n... [{len(content) - 5000} more characters] ...")

---
## 3. Beam Evaluation & Interpolation <a id='3-beam'></a>

### 3.1 Overview

The Jones matrix $\mathbf{J}_p(\hat{\mathbf{s}}, \nu)$ encodes the **primary beam** of antenna $p$. Physically, the beam describes how the antenna converts incoming electromagnetic radiation from direction $\hat{\mathbf{s}}$ into output voltage at its feed terminals. It is direction-dependent, frequency-dependent, and polarization-dependent.

For a dipole-like antenna (such as HERA's crossed dipoles), the beam has a broad main lobe centered on the zenith — the antenna is most sensitive to sources directly overhead — and sensitivity gradually decreases toward the horizon.

`matvis` supports multiple beam types through a common interface.

### 3.2 Beam Types Supported

1. **UVBeam** (from `pyuvdata`): Tabulated beam data on a regular $(\theta, \phi)$ grid at discrete frequencies. These are typically generated from electromagnetic simulations (CST, FEKO) or derived from holographic measurements. Evaluated at arbitrary directions via interpolation.

2. **AnalyticBeam**: Beams defined by a simple mathematical function. Useful for testing and for cases where a detailed beam model is unnecessary. Common types: Gaussian, Airy disk, uniform/isotropic.

3. **Per-antenna beams**: Each antenna can have its own distinct beam pattern (e.g., to model manufacturing variations). Alternatively, a single shared beam can be used for all antennas (the most common case, since identical antennas have identical beams).

### 3.3 Beam Evaluation Coordinates

For each above-horizon source $s$ at time $t$, the beam is evaluated at the source's topocentric position expressed in spherical coordinates:

$$\theta_s = \arccos(n_s)$$
$$\phi_s = \arctan2(l_s, m_s)$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $\theta_s$ | Zenith angle of source $s$: $0$ at zenith, $\pi/2$ at the horizon. |
> | $\phi_s$ | Azimuth of source $s$: $0$ = North, $\pi/2$ = East. |
> | $l_s, m_s, n_s$ | ENU direction cosines (East, North, Up) of source $s$ at the current time step. |
> | $(e_s, n_s, u_s)$ | Alternative notation for the same ENU components: $e_s = l_s$ (East), $n_s = m_s$ (North), $u_s = n_s$ (Up). |

**Azimuth convention:** `matvis` uses `arctan2(East, North)`, measuring azimuth **East from North** (standard IAU astronomical convention: N=0°, E=90°, S=180°, W=270°).

### 3.4 Beam Interpolation (UVBeam)

For `UVBeam` objects, the beam is stored on a regular angular grid and must be interpolated to the exact source positions:

$$J_p^{\alpha\gamma}(\theta_s, \phi_s, \nu_k) = \text{interp}\left(\text{beam\_data}, \theta_s, \phi_s, \nu_k\right)$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $J_p^{\alpha\gamma}$ | One element of the Jones matrix: "How much voltage does feed $\alpha$ on antenna $p$ produce in response to sky polarization component $\gamma$, from a wave arriving from direction $(\theta_s, \phi_s)$ at frequency $\nu_k$?" |
> | $p$ | Antenna index. |
> | $\alpha$ | **Antenna feed** index: $\alpha \in \{X, Y\}$. $X$ is one physical dipole (e.g., E–W oriented), $Y$ is the orthogonal dipole (N–S oriented). |
> | $\gamma$ | **Sky polarization** index: $\gamma \in \{\hat{\theta}, \hat{\phi}\}$. $\hat{\theta}$ is the unit vector on the sky tangent sphere pointing in the zenith-angle direction; $\hat{\phi}$ is the unit vector pointing in the azimuthal direction. These describe the orientation of the incoming electric field. |
> | $\theta_s, \phi_s$ | Zenith angle and azimuth of source $s$ (defined above). |
> | $\nu_k$ | Frequency of channel $k$. |

The interpolation is typically **bilinear in angle** (linear in both $\theta$ and $\phi$) and can be **linear or cubic spline in frequency**.

**Shape of the beam Jones matrix:** For a dual-polarization feed (2 feeds $\times$ 2 sky polarizations):

$$\mathbf{J}_p = \begin{pmatrix} J_p^{X\hat{\theta}} & J_p^{X\hat{\phi}} \\ J_p^{Y\hat{\theta}} & J_p^{Y\hat{\phi}} \end{pmatrix}$$

> **Index legend for this matrix:**
>
> | Element | Meaning |
> |---------|---------|
> | $J_p^{X\hat{\theta}}$ | "Co-polar" response of the $X$-feed to the $\hat{\theta}$ sky component. For a well-designed antenna, this is the dominant term for the $X$-feed. |
> | $J_p^{X\hat{\phi}}$ | "Cross-polar" leakage: how much the $X$-feed responds to the $\hat{\phi}$ sky component. Ideally small. |
> | $J_p^{Y\hat{\theta}}$ | Cross-polar leakage of the $Y$-feed. Ideally small. |
> | $J_p^{Y\hat{\phi}}$ | Co-polar response of the $Y$-feed to the $\hat{\phi}$ sky component. |

**Physical picture:** An incoming electromagnetic wave from direction $\hat{\mathbf{s}}$ has electric field components $(E^{\hat{\theta}}, E^{\hat{\phi}})$. The antenna's two feeds produce voltages:

$$\begin{pmatrix} v^X \\ v^Y \end{pmatrix} = \begin{pmatrix} J^{X\hat{\theta}} & J^{X\hat{\phi}} \\ J^{Y\hat{\theta}} & J^{Y\hat{\phi}} \end{pmatrix} \begin{pmatrix} E^{\hat{\theta}} \\ E^{\hat{\phi}} \end{pmatrix}$$

### 3.5 Analytic Beam Evaluation

For analytic beams (e.g., Gaussian):

**Gaussian beam:**
$$A(\theta) = \exp\left(-\frac{\theta^2}{2\sigma^2}\right)$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $A(\theta)$ | Beam amplitude (E-field, not power) at zenith angle $\theta$. This is assumed azimuthally symmetric (same in all directions at the same $\theta$). |
> | $\theta$ | Zenith angle (radians). |
> | $\sigma$ | Gaussian width parameter: $\sigma = \text{FWHM} / (2\sqrt{2 \ln 2})$, where FWHM is the Full Width at Half Maximum of the **power** beam $|A|^2$. |
> | FWHM | Full Width at Half Maximum of the power beam. Can be frequency-dependent; typically $\text{FWHM} \propto \lambda / D \propto 1/\nu$, so the beam narrows at higher frequencies. |

**Airy beam** (diffraction pattern of a uniformly illuminated circular aperture):
$$A(\theta) = \left(\frac{2 J_1(\pi D \sin\theta / \lambda)}{\pi D \sin\theta / \lambda}\right)^2$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $J_1$ | Bessel function of the first kind, order 1 (**not** a Jones matrix — unfortunate notation collision). |
> | $D$ | Diameter of the dish/aperture (meters). For HERA dishes: $D \approx 14$ m. |
> | $\lambda$ | Wavelength: $\lambda = c / \nu$. At 150 MHz: $\lambda \approx 2$ m. |

### 3.6 Beam as Applied to Sources

After evaluation, the beam Jones matrix is applied per-antenna, per-source, per-frequency. The result is a complex-valued Jones matrix for each (antenna, source, frequency) triple:

$$\mathbf{J}_{p,s}(\nu_k) \in \mathbb{C}^{2 \times 2}$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $\mathbf{J}_{p,s}(\nu_k)$ | The $2 \times 2$ Jones matrix of antenna $p$, evaluated in the direction of source $s$, at frequency $\nu_k$. |
> | $p$ | Antenna index. |
> | $s$ | Source index. |
> | $\nu_k$ | Frequency of channel $k$. |

In the **unpolarized (Stokes I only) case**, the full $2 \times 2$ Jones matrix is not needed. Instead, only a single scalar beam value per feed is used — typically the dominant diagonal element (e.g., $J^{X\hat{\theta}}$ for the $X$-feed). This "scalar beam" approximation ignores cross-polarization leakage entirely and treats the beam as one complex number per (antenna, direction, frequency).

In [ ]:
# Let's examine the beam handling code
beam_files = glob.glob(os.path.join(matvis_path, '**/beam*.py'), recursive=True)
for f in sorted(beam_files):
    print(f"\n{'='*80}")
    print(f"FILE: {f}")
    print(f"{'='*80}")
    with open(f, 'r') as fh:
        content = fh.read()
        print(content[:8000])
        if len(content) > 8000:
            print(f"\n... [{len(content) - 8000} more characters] ...")

---
## 4. Beam Normalization: Peak vs. Area <a id='4-beam-norm'></a>

### 4.1 The Critical Question: How is the Beam Normalized?

**`matvis` uses PEAK normalization, NOT area normalization.**

The beam is normalized so that its maximum value (at zenith) is unity:

$$\max_{\hat{\mathbf{s}}} \left| J^{\alpha\gamma}(\hat{\mathbf{s}}, \nu) \right|^2 = 1$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $\hat{\mathbf{s}}$ | Direction on the sky (searched over to find the maximum). |
> | $J^{\alpha\gamma}(\hat{\mathbf{s}}, \nu)$ | Jones element for feed $\alpha$, sky polarization $\gamma$, direction $\hat{\mathbf{s}}$, frequency $\nu$. |
> | $\alpha$ | Feed polarization index ($X$ or $Y$). |
> | $\gamma$ | Sky polarization index ($\hat{\theta}$ or $\hat{\phi}$). |
> | $\nu$ | Frequency. |

This means at zenith ($\theta = 0$), the beam power is unity, and the beam is **not** normalized by its integrated solid angle.

### 4.2 Implementation Details

`matvis` performs beam normalization in these steps:

1. The beam is evaluated for all above-horizon source directions at the current frequency $\nu_k$.
2. A **peak normalization factor** is computed by evaluating the beam at zenith ($\theta = 0$, $\phi = 0$).
3. All beam values are divided by this zenith value.

The normalization formula:

$$\tilde{J}_p^{\alpha\gamma}(\hat{\mathbf{s}}_s, \nu) = \frac{J_p^{\alpha\gamma}(\hat{\mathbf{s}}_s, \nu)}{J_p^{\alpha\gamma}(\hat{\mathbf{z}}, \nu)}$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $\tilde{J}_p^{\alpha\gamma}$ | The **normalized** Jones element. The tilde ($\sim$) denotes that peak normalization has been applied. |
> | $J_p^{\alpha\gamma}(\hat{\mathbf{s}}_s, \nu)$ | **Unnormalized** (raw) Jones element of antenna $p$, feed $\alpha$, sky polarization $\gamma$, evaluated toward source $s$ at frequency $\nu$. |
> | $J_p^{\alpha\gamma}(\hat{\mathbf{z}}, \nu)$ | The beam value **at zenith** ($\hat{\mathbf{z}}$ means $\theta = 0$). This is the normalization denominator. Computed once per frequency, per feed, per antenna (or once total if all antennas share a beam). |
> | $p$ | Antenna index. |
> | $\alpha$ | Feed index ($X$ or $Y$). |
> | $\gamma$ | Sky polarization index ($\hat{\theta}$ or $\hat{\phi}$). |
> | $s$ | Source index. |

**Why peak normalization?**
- Source catalogs list flux densities in **Jy** (Jansky). These values assume that the instrument's peak response is calibrated to unity.
- With peak normalization, a point source of flux $S$ Jy at the zenith contributes exactly $S$ Jy to the visibility — the beam just passes it through at full strength.
- Away from zenith, the source's apparent contribution is attenuated: $\tilde{J}^2 \cdot S$ Jy. This gives visibilities directly in **Jy** units.
- This is the standard convention in radio interferometry and is consistent with how calibrator fluxes are defined.

### 4.3 Contrast with Area Normalization

Area normalization instead defines the beam by its integrated solid angle:

$$\Omega_B = \int |A(\hat{\mathbf{s}})|^2 \, d\Omega$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $\Omega_B$ | **Beam solid angle**: the integral of the power beam over the full sky. Units: steradians. Typical value for HERA at 150 MHz: $\sim 0.5$ sr. |
> | $A(\hat{\mathbf{s}})$ | Beam voltage amplitude in direction $\hat{\mathbf{s}}$. $|A|^2$ is the power beam. |
> | $d\Omega$ | Differential solid angle $= \sin\theta \, d\theta \, d\phi$. |

If area normalization were used, the visibility units would involve steradians and would not directly represent Jy. **matvis does NOT use area normalization.** The beam solid angle only enters when converting between flux density (Jy) and brightness temperature (K), via $T_b = \lambda^2 S / (2 k_B \Omega_B)$.

### 4.4 The `beam_normalize` Flag

In `matvis`, the normalization step is controlled by a parameter. When enabled:

$$J_{\text{normalized}}(\hat{\mathbf{s}}) = \frac{J(\hat{\mathbf{s}})}{J(\hat{\mathbf{z}})}$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $J_{\text{normalized}}$ | Peak-normalized beam. |
> | $J(\hat{\mathbf{z}})$ | Beam value at zenith (the peak). |

This ensures:
- $J_{\text{normalized}}(\hat{\mathbf{z}}) = 1$ (beam is unity at zenith)
- Power beam at zenith: $|J_{\text{normalized}}(\hat{\mathbf{z}})|^2 = 1$
- Visibilities are directly in the same units as the input source fluxes (typically Jy)

**Caveat:** If the beam model already provides a peak-normalized beam (zenith value = 1), applying normalization again is redundant but harmless (divides by 1). If the beam model provides an unnormalized beam (e.g., raw electromagnetic simulation output including effective area factors), normalization is essential for correct flux scaling.

In [ ]:
# Let's find and display the exact normalization code
import re

# Search for normalization-related code across all files
all_py_files = glob.glob(os.path.join(matvis_path, '**/*.py'), recursive=True)

for f in sorted(all_py_files):
    with open(f, 'r') as fh:
        content = fh.read()
    
    # Search for normalization patterns
    patterns = ['normalize', 'norm', 'zenith', 'peak', 'beam_val']
    for pattern in patterns:
        matches = [(i, line) for i, line in enumerate(content.split('\n')) 
                   if pattern.lower() in line.lower() and not line.strip().startswith('#')]
        if matches:
            relpath = os.path.relpath(f, matvis_path)
            for line_num, line in matches:
                print(f"{relpath}:{line_num+1}: {line.strip()}")

---
## 5. Source Coherency Matrix Construction <a id='5-coherency'></a>

### 5.1 Stokes to Coherency Conversion

The sky model specifies each source's brightness in terms of **Stokes parameters** $(I, Q, U, V)$. These are four real numbers that fully describe the intensity and polarization state of the radiation. The **coherency matrix** is a $2 \times 2$ Hermitian matrix that encodes the same information in the form needed by the RIME:

$$\mathbf{C}_s = \begin{pmatrix} I_s + Q_s & U_s + iV_s \\ U_s - iV_s & I_s - Q_s \end{pmatrix}$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $\mathbf{C}_s$ | The $2 \times 2$ coherency matrix of source $s$. Its rows and columns are indexed by sky polarizations $(\gamma, \delta) \in \{\hat{\theta}, \hat{\phi}\}$. |
> | $s$ | Source index ($s = 1, \ldots, N_{\text{src}}$). |
> | $I_s$ | **Stokes $I$**: total intensity (total flux density) of source $s$. Always $\geq 0$ for physical sources. Units: Jy. |
> | $Q_s$ | **Stokes $Q$**: linear polarization along/perpendicular to the reference axis. $Q > 0$ means more power in the $\hat{\theta}$ direction than in the $\hat{\phi}$ direction. |
> | $U_s$ | **Stokes $U$**: linear polarization at $\pm 45°$ to the reference axis. |
> | $V_s$ | **Stokes $V$**: circular polarization. $V > 0$ is right-hand circular. |
> | $i$ | The imaginary unit $\sqrt{-1}$ (not an index here). |

**Physical meaning of the Stokes parameters:**
- $I$ = total power (sum of power in all polarization states). This is what we usually call "the flux."
- $Q$ and $U$ together describe **linear polarization**: a preferred orientation of the electric field oscillation. The **polarization fraction** is $p = \sqrt{Q^2+U^2+V^2}/I$, and the **polarization angle** is $\chi = \frac{1}{2}\arctan(U/Q)$.
- $V$ = circular polarization: the electric field traces an ellipse (or circle) rather than oscillating in a fixed plane.
- For an **unpolarized** source: $Q = U = V = 0$, and $p = 0$.
- Physical constraint: $I^2 \geq Q^2 + U^2 + V^2$ (equality for 100% polarized light).

**Note on the factor of $1/2$:** Some textbooks define $\mathbf{C}_s = \frac{1}{2}(\ldots)$. `matvis` follows the convention **without** this extra factor, so $\mathbf{C}_s$ has the same units as the Stokes parameters (Jy).

### 5.2 Unpolarized (Stokes I only) Case

For unpolarized sources ($Q = U = V = 0$), the coherency matrix simplifies to:

$$\mathbf{C}_s = I_s \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix} = I_s \, \mathbb{I}_2$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $I_s$ | Stokes I flux density of source $s$ (scalar, in Jy). |
> | $\mathbb{I}_2$ | The $2 \times 2$ identity matrix. |

When this is inserted into the RIME, the sum over sky polarization indices $(\gamma, \delta)$ simplifies because the identity commutes with everything:

$$V_{pq}^{\alpha\beta}(\nu) = \sum_{s} I_s(\nu) \; J_p^{\alpha}(\hat{\mathbf{s}}_s, \nu) \; J_q^{\beta *}(\hat{\mathbf{s}}_s, \nu) \; e^{-2\pi i \nu \, \mathbf{b}_{pq} \cdot \hat{\mathbf{s}}_s / c}$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $V_{pq}^{\alpha\beta}$ | Visibility for baseline $(p, q)$, feed pair $(\alpha, \beta)$. |
> | $p, q$ | Antenna indices. |
> | $\alpha$ | Feed of antenna $p$ ($X$ or $Y$). |
> | $\beta$ | Feed of antenna $q$ ($X$ or $Y$). |
> | $s$ | Source index (summed over). |
> | $I_s(\nu)$ | Flux density of source $s$ at frequency $\nu$. |
> | $J_p^{\alpha}$ | Scalar beam value for feed $\alpha$ of antenna $p$ in the direction of source $s$. In the Stokes I mode, $\gamma$ is implicit — we use just the dominant diagonal Jones component per feed. |
> | $J_q^{\beta *}$ | Complex conjugate of the beam for feed $\beta$ of antenna $q$. |

This formula shows that in the unpolarized case, each source contributes independently, weighted by its flux $I_s$, the beam responses at both antennas, and the geometric fringe factor.

### 5.3 The Square Root Trick

To factorize the RIME into per-antenna quantities, `matvis` absorbs the source flux into the per-antenna beam-weighted vectors:

$$\sqrt{I_s(\nu)}$$

is multiplied into each antenna's vector. When the outer product $\mathbf{A} \mathbf{A}^H$ is taken, the product $\sqrt{I_s} \times \sqrt{I_s} = I_s$ is recovered.

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $\sqrt{I_s(\nu)}$ | Positive real square root of the flux density of source $s$ at frequency $\nu$. Units: $\sqrt{\text{Jy}}$. |
> | $s$ | Source index. |

**Why the square root?** The visibility is a *bilinear* function (one factor from antenna $p$, one from antenna $q$). To express it as an outer product of per-antenna vectors, each antenna must carry $\sqrt{I_s}$ so that $\sqrt{I_s} \times \sqrt{I_s} = I_s$. This only works for $I_s \geq 0$. Negative fluxes (which can arise in residual/subtracted maps) would require a different formulation.

### 5.4 Flux Density Spectrum

Source flux densities can be frequency-dependent (sources have spectra). `matvis` accepts:
- A **1D array** of shape $(N_{\text{src}},)$ for frequency-independent flux
- A **2D array** of shape $(N_{\text{freq}}, N_{\text{src}})$ for frequency-dependent flux densities

At frequency channel $\nu_k$:
$$I_s(\nu_k) = \text{flux\_array}[k, s]$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $k$ | Frequency channel index, $k \in \{0, 1, \ldots, N_{\text{freq}} - 1\}$. |
> | $s$ | Source index, $s \in \{0, 1, \ldots, N_{\text{src}} - 1\}$. |
> | $I_s(\nu_k)$ | Flux density (Jy) of source $s$ at frequency $\nu_k$. |

A typical spectral model is a power law: $I_s(\nu) = I_{s,0} (\nu / \nu_0)^{\alpha_{\text{spec}}}$, where $\alpha_{\text{spec}}$ is the spectral index (not to be confused with the feed polarization index $\alpha$). Steep-spectrum synchrotron sources have $\alpha_{\text{spec}} \approx -0.7$ to $-1.0$ (brighter at lower frequencies).

In [ ]:
# Search for coherency/Stokes/flux handling
for f in sorted(all_py_files):
    with open(f, 'r') as fh:
        content = fh.read()
    
    patterns = ['sqrt', 'coherency', 'stokes', 'flux', 'I_sky', 'sky_flux', 'Isqrt', 'sqrtI']
    found_lines = []
    for pattern in patterns:
        matches = [(i, line) for i, line in enumerate(content.split('\n')) 
                   if pattern.lower() in line.lower() 
                   and not line.strip().startswith('#')
                   and len(line.strip()) > 5]
        found_lines.extend(matches)
    
    if found_lines:
        relpath = os.path.relpath(f, matvis_path)
        seen = set()
        for line_num, line in sorted(set(found_lines)):
            key = line.strip()
            if key not in seen:
                print(f"{relpath}:{line_num+1}: {key}")
                seen.add(key)

---
## 6. Fringe (Phase) Computation <a id='6-fringe'></a>

### 6.1 The Geometric Phase Term

The "fringe" encodes the **path length difference** between a wavefront arriving at one antenna versus another. For a plane wave from direction $\hat{\mathbf{s}}_s$, the extra distance travelled to reach antenna $p$ (compared to the phase center at the array origin) is $\hat{\mathbf{s}}_s \cdot \mathbf{r}_p$. This translates into a phase:

$$\phi_{p,s}(\nu) = -2\pi \frac{\nu}{c} \, \hat{\mathbf{s}}_s \cdot \mathbf{r}_p$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $\phi_{p,s}(\nu)$ | Geometric phase (in radians) accumulated at antenna $p$ due to source $s$ at frequency $\nu$. |
> | $p$ | Antenna index. |
> | $s$ | Source index. |
> | $\nu$ | Frequency (Hz). |
> | $c$ | Speed of light ($2.998 \times 10^8$ m/s). |
> | $\hat{\mathbf{s}}_s$ | Unit direction vector toward source $s$ in the topocentric ENU frame. Components: $(l_s, m_s, n_s)$. |
> | $\mathbf{r}_p$ | Position vector of antenna $p$ in the topocentric ENU frame. Components: $(E_p, N_p, U_p)$ in meters. |
> | $\hat{\mathbf{s}}_s \cdot \mathbf{r}_p$ | Dot product $= l_s E_p + m_s N_p + n_s U_p$: the projected distance (meters) from the phase center to antenna $p$ along the direction of source $s$. |

The per-antenna fringe phasor is then:

$$e^{i\phi_{p,s}(\nu)} = \exp\left(-2\pi i \frac{\nu}{c} \hat{\mathbf{s}}_s \cdot \mathbf{r}_p\right)$$

**Physical intuition:** For a flat array (all antennas at the same height, $U_p = 0$) and a source at the zenith ($\hat{\mathbf{s}} = \hat{\mathbf{z}} = (0, 0, 1)$), the dot product $\hat{\mathbf{s}} \cdot \mathbf{r}_p = U_p = 0$ for all antennas. So a zenith source produces **no** phase difference between antennas — the wavefront arrives everywhere simultaneously. For a source off-zenith, different antennas "see" the wavefront at different times, producing a direction-dependent phase.

### 6.2 Factorization into Per-Antenna Phases

The baseline fringe from the RIME can be split into a product of per-antenna phases:

$$e^{-2\pi i \nu \mathbf{b}_{pq} \cdot \hat{\mathbf{s}}_s / c} = e^{-2\pi i \nu \hat{\mathbf{s}}_s \cdot \mathbf{r}_p / c} \cdot e^{+2\pi i \nu \hat{\mathbf{s}}_s \cdot \mathbf{r}_q / c}$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $\mathbf{b}_{pq} = \mathbf{r}_p - \mathbf{r}_q$ | Baseline vector: the vector separation from antenna $q$ to antenna $p$ (meters). |
> | $e^{-2\pi i \nu \hat{\mathbf{s}}_s \cdot \mathbf{r}_p / c}$ | Phase phasor for antenna $p$ toward source $s$. |
> | $e^{+2\pi i \nu \hat{\mathbf{s}}_s \cdot \mathbf{r}_q / c}$ | Conjugated phase phasor for antenna $q$. The $+$ sign arises from complex conjugation of the second antenna's contribution in $\mathbf{J}_q^H$. |

This factorization is **critical** for the matrix multiplication approach: each antenna carries its own phase, and the baseline phase (which depends on the *difference* $\mathbf{r}_p - \mathbf{r}_q$) emerges naturally from the outer product $A_p \cdot A_q^*$.

### 6.3 Numerical Implementation: Geometric Delay $\tau$

`matvis` computes the **geometric delay** $\tau_{p,s}$ (in seconds) first:

$$\tau_{p,s} = \frac{\hat{\mathbf{s}}_s \cdot \mathbf{r}_p}{c}$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $\tau_{p,s}$ | Geometric delay for antenna $p$ toward source $s$. Units: seconds. This is the time light takes to travel the projected distance $\hat{\mathbf{s}}_s \cdot \mathbf{r}_p$. Typical values: $\sim 1\,\mu\text{s}$ for a 300 m baseline. |
> | $p$ | Antenna index. |
> | $s$ | Source index. |

This delay is **frequency-independent** — it depends only on geometry. Then the phase at each frequency is:

$$\phi_{p,s}(\nu_k) = -2\pi \nu_k \tau_{p,s}$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $\nu_k$ | Frequency of channel $k$ (Hz). |
> | $\tau_{p,s}$ | Geometric delay (seconds), defined above. |

**As a matrix operation:**

$$\boldsymbol{\tau} = \frac{1}{c} \, \mathbf{R}_{\text{ant}} \cdot \hat{\mathbf{S}}_{\text{top}}^T$$

> **Index legend:**
>
> | Symbol | Shape | Meaning |
> |--------|-------|---------|
> | $\boldsymbol{\tau}$ | $(N_{\text{ant}}, N_{\text{src}})$ | Matrix of geometric delays. Element $(p, s)$ is $\tau_{p,s}$. |
> | $\mathbf{R}_{\text{ant}}$ | $(N_{\text{ant}}, 3)$ | Antenna position matrix. Row $p$ is $\mathbf{r}_p = (E_p, N_p, U_p)$ in meters. |
> | $\hat{\mathbf{S}}_{\text{top}}$ | $(N_{\text{src}}, 3)$ | Topocentric source direction matrix. Row $s$ is $\hat{\mathbf{s}}_s = (l_s, m_s, n_s)$. |

### 6.4 Complex Exponential Computation

The fringe phasor for each (antenna, source) pair at frequency $\nu_k$:

$$\text{fringe}_{p,s}(\nu_k) = e^{-2\pi i \nu_k \tau_{p,s}}$$

In code:
```python
tau = (antpos @ crd_top) / speed_of_light  # shape (Nant, Nsrc)
fringe = np.exp(-2j * np.pi * freq * tau)   # shape (Nant, Nsrc)
```

**Key optimization:** `matvis` computes $\tau$ **once per time step** (since source and antenna positions don't depend on frequency) and reuses it across all frequency channels. Only the complex exponential $e^{-2\pi i \nu_k \tau}$ is recomputed at each frequency — this is a much cheaper element-wise operation.

**Numerical note on phase wrapping:** The phase $2\pi\nu\tau$ can be very large. For $\nu = 150$ MHz and $\tau = 1\,\mu$s: $2\pi \times 1.5 \times 10^8 \times 10^{-6} \approx 940$ radians ($\approx 150$ full turns). The `np.exp` function handles this correctly, but if intermediate values are stored in float32 (single precision), phase accuracy degrades for such large arguments. This is one reason `matvis` defaults to float64.

In [ ]:
# Search for fringe/phase/tau computation
for f in sorted(all_py_files):
    with open(f, 'r') as fh:
        content = fh.read()
    
    patterns = ['tau', 'fringe', 'exp(', 'phase', 'antpos', 'crd_top']
    found_lines = []
    for pattern in patterns:
        for i, line in enumerate(content.split('\n')):
            if pattern in line and not line.strip().startswith('#') and len(line.strip()) > 5:
                found_lines.append((i, line))
    
    if found_lines:
        relpath = os.path.relpath(f, matvis_path)
        seen = set()
        for line_num, line in sorted(set(found_lines)):
            key = line.strip()
            if key not in seen:
                print(f"{relpath}:{line_num+1}: {key}")
                seen.add(key)

---
## 7. The Core Matrix Multiplication (Visibility Formation) <a id='7-core-multiplication'></a>

### 7.1 Construction of the Per-Antenna Vector

For each antenna $p$, feed polarization $\alpha$, frequency $\nu_k$, and each source $s$, the per-antenna "measurement" scalar is:

$$A_p^{\alpha}(s, \nu_k) = \tilde{J}_p^{\alpha}(\hat{\mathbf{s}}_s, \nu_k) \cdot \sqrt{I_s(\nu_k)} \cdot e^{-2\pi i \nu_k \tau_{p,s}}$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $A_p^{\alpha}(s, \nu_k)$ | A single complex number: the combined beam × flux × fringe contribution from source $s$ to antenna $p$'s feed $\alpha$ at frequency $\nu_k$. |
> | $p$ | Antenna index ($p = 1, \ldots, N_{\text{ant}}$). |
> | $\alpha$ | Feed polarization ($\alpha \in \{X, Y\}$): which dipole on antenna $p$. |
> | $s$ | Source index ($s = 1, \ldots, N_{\text{src}}$). |
> | $\nu_k$ | Frequency of channel $k$. |
> | $\tilde{J}_p^{\alpha}(\hat{\mathbf{s}}_s, \nu_k)$ | Peak-normalized beam response: complex-valued, includes both amplitude and phase of the beam pattern. In the Stokes I scalar-beam mode, this is a single number (the dominant diagonal Jones element for feed $\alpha$). |
> | $\sqrt{I_s(\nu_k)}$ | Real positive square root of the flux density of source $s$. |
> | $e^{-2\pi i \nu_k \tau_{p,s}}$ | Fringe phasor: encodes the geometric delay of antenna $p$ toward source $s$. |
> | $\tau_{p,s}$ | Geometric delay (seconds), defined in Section 6. |

This combines three factors:
1. **Beam response** $\tilde{J}_p^{\alpha}$: complex-valued, direction- and frequency-dependent antenna sensitivity
2. **Source amplitude** $\sqrt{I_s}$: real-valued, encodes "how bright" the source is (split for outer-product factorization)
3. **Fringe phasor** $e^{-2\pi i \nu_k \tau_{p,s}}$: complex-valued geometric phase

### 7.2 The Key Matrix Product

Collect all per-antenna scalars (for one feed $\alpha$ and one frequency $\nu_k$) into a matrix:

$$\mathbf{A}^{\alpha}(\nu_k) \in \mathbb{C}^{N_{\text{ant}} \times N_{\text{src}}}$$

> **Index legend:**
>
> | Symbol | Shape | Meaning |
> |--------|-------|---------|
> | $\mathbf{A}^{\alpha}(\nu_k)$ | $(N_{\text{ant}}, N_{\text{src}})$ | The "measurement matrix" for feed $\alpha$ at frequency $\nu_k$. Row $p$ contains the contributions from all sources to antenna $p$. Column $s$ contains how source $s$ appears to all antennas. |
> | Element $[\mathbf{A}^\alpha]_{p,s}$ | scalar | $= A_p^{\alpha}(s, \nu_k)$. |

The visibility matrix for feed pair $(\alpha, \beta)$ is:

$$\mathbf{V}^{\alpha\beta}(\nu_k) = \mathbf{A}^{\alpha}(\nu_k) \cdot \left[\mathbf{A}^{\beta}(\nu_k)\right]^H$$

> **Index legend:**
>
> | Symbol | Shape | Meaning |
> |--------|-------|---------|
> | $\mathbf{V}^{\alpha\beta}(\nu_k)$ | $(N_{\text{ant}}, N_{\text{ant}})$ | Visibility matrix for feed pair $(\alpha, \beta)$ at frequency $\nu_k$. Element $(p, q)$ is the visibility for baseline $pq$. |
> | $\alpha$ | — | Feed of the "first" antenna (the row of $\mathbf{V}$). |
> | $\beta$ | — | Feed of the "second" antenna (the column of $\mathbf{V}$, conjugated via $^H$). |
> | $[\mathbf{A}^\beta]^H$ | $(N_{\text{src}}, N_{\text{ant}})$ | Conjugate-transpose: every element is complex-conjugated and the matrix is transposed. |

This is a **matrix multiplication** of shapes: $(N_{\text{ant}}, N_{\text{src}}) \times (N_{\text{src}}, N_{\text{ant}}) \Rightarrow (N_{\text{ant}}, N_{\text{ant}})$.

**Special property when $\alpha = \beta$:** The product $\mathbf{A} \mathbf{A}^H$ is a Hermitian matrix ($V_{pq} = V_{qp}^*$). This reflects the interferometric conjugation symmetry: swapping the two antennas conjugates the visibility.

### 7.3 Expansion of the Matrix Product

Element $(p, q)$ of the result:

$$V_{pq}^{\alpha\beta}(\nu_k) = \sum_{s=1}^{N_{\text{src}}} A_p^{\alpha}(s, \nu_k) \cdot A_q^{\beta *}(s, \nu_k)$$

> **Index legend:** $p, q$ = antenna indices; $\alpha, \beta$ = feed indices; $s$ = source index summed over; $*$ = complex conjugation.

Expanding the definition of $A$:

$$= \sum_{s=1}^{N_{\text{src}}} \tilde{J}_p^{\alpha}(\hat{\mathbf{s}}_s, \nu_k) \cdot \sqrt{I_s} \cdot e^{-2\pi i \nu_k \tau_{p,s}} \cdot \tilde{J}_q^{\beta *}(\hat{\mathbf{s}}_s, \nu_k) \cdot \sqrt{I_s} \cdot e^{+2\pi i \nu_k \tau_{q,s}}$$

> Note: $A_q^{\beta *}$ introduces both a conjugation of $\tilde{J}_q^\beta$ (giving $\tilde{J}_q^{\beta *}$) and a sign flip in the exponential (giving $e^{+2\pi i \nu_k \tau_{q,s}}$). The $\sqrt{I_s}$ is real, so its conjugate is itself.

Collecting terms:

$$= \sum_{s=1}^{N_{\text{src}}} I_s(\nu_k) \; \tilde{J}_p^{\alpha}(\hat{\mathbf{s}}_s, \nu_k) \; \tilde{J}_q^{\beta *}(\hat{\mathbf{s}}_s, \nu_k) \; e^{-2\pi i \nu_k (\tau_{p,s} - \tau_{q,s})}$$

Since $\tau_{p,s} - \tau_{q,s} = \hat{\mathbf{s}}_s \cdot (\mathbf{r}_p - \mathbf{r}_q)/c = \hat{\mathbf{s}}_s \cdot \mathbf{b}_{pq}/c$:

$$= \sum_{s=1}^{N_{\text{src}}} I_s(\nu_k) \; \tilde{J}_p^{\alpha}(\hat{\mathbf{s}}_s, \nu_k) \; \tilde{J}_q^{\beta *}(\hat{\mathbf{s}}_s, \nu_k) \; e^{-2\pi i \nu_k \mathbf{b}_{pq} \cdot \hat{\mathbf{s}}_s / c}$$

**This is exactly the discrete RIME for unpolarized sources!** ✓ The matrix multiplication automatically produces the correct per-baseline result.

### 7.4 Computational Complexity

| Operation | Complexity | Indices looped over |
|-----------|------------|---------------------|
| Coordinate transform | $O(N_{\text{src}})$ per time step | $s$ |
| Beam evaluation | $O(N_{\text{ant}} \times N_{\text{src}})$ per freq per time | $p, s$ |
| Fringe computation | $O(N_{\text{ant}} \times N_{\text{src}})$ per freq per time | $p, s$ |
| Matrix multiply $\mathbf{A}\mathbf{A}^H$ | $O(N_{\text{ant}}^2 \times N_{\text{src}})$ per freq per time | $p, q, s$ |
| **Total** | $O(N_{\text{time}} \times N_{\text{freq}} \times N_{\text{ant}}^2 \times N_{\text{src}})$ | $t, k, p, q, s$ |

The matrix multiplication dominates and is highly optimized via BLAS (Basic Linear Algebra Subprograms, e.g., Intel MKL, OpenBLAS) on CPU or cuBLAS on GPU. These libraries exploit CPU cache, SIMD vector instructions, and multi-threading.

### 7.5 Why This Works: The Factorizability of the RIME

The full RIME for the polarized case involves:
$$\mathbf{V}_{pq} = \sum_s \mathbf{J}_p \mathbf{C}_s \mathbf{J}_q^H \cdot e^{i\phi_{pq,s}}$$

> **Index legend:** $p, q$ = antennas; $s$ = source index; $\mathbf{J}_p$ = $2\times 2$ Jones matrix (indices $\alpha\gamma$); $\mathbf{C}_s$ = $2 \times 2$ coherency matrix (indices $\gamma\delta$); $\mathbf{J}_q^H$ = conjugate-transposed Jones (indices $\delta\beta$); $\phi_{pq,s}$ = baseline geometric phase.

For Stokes I only ($\mathbf{C}_s = I_s \mathbb{I}$), the identity commutes with everything, so the product $\mathbf{J}_p (I_s \mathbb{I}) \mathbf{J}_q^H = I_s \mathbf{J}_p \mathbf{J}_q^H$ factorizes cleanly into per-antenna contributions. This allows the $\sqrt{I_s}$ splitting and the $\mathbf{A}\mathbf{A}^H$ formulation.

For the fully polarized case ($Q, U, V \neq 0$), the coherency matrix is not proportional to the identity and does not commute. `matvis` handles this by constructing separate $\mathbf{A}$ matrices for each feed–sky polarization pair $(\alpha, \gamma)$ and accumulating the contributions.

In [ ]:
# Let's find and print the actual core computation loop
# This is the most important part of the code
for f in sorted(all_py_files):
    with open(f, 'r') as fh:
        content = fh.read()
    
    # Look for the matrix multiplication
    if 'matmul' in content or 'einsum' in content or '@' in content or '.dot(' in content:
        relpath = os.path.relpath(f, matvis_path)
        lines = content.split('\n')
        for i, line in enumerate(lines):
            if any(kw in line for kw in ['matmul', 'einsum', ' @ ', '.dot(', 'outer']):
                if not line.strip().startswith('#'):
                    # Print surrounding context
                    start = max(0, i-3)
                    end = min(len(lines), i+4)
                    print(f"\n--- {relpath}:{i+1} ---")
                    for j in range(start, end):
                        marker = '>>>' if j == i else '   '
                        print(f"{marker} {j+1}: {lines[j]}")

---
## 8. Polarization Handling & Feed Mapping <a id='8-polarization'></a>

### 8.1 Full Polarization RIME

For a dual-polarization system (such as HERA, where each antenna element has two crossed dipole feeds), the Jones matrix is a full $2 \times 2$ complex matrix:

$$\mathbf{J}_p = \begin{pmatrix} J_p^{X\hat{\theta}} & J_p^{X\hat{\phi}} \\ J_p^{Y\hat{\theta}} & J_p^{Y\hat{\phi}} \end{pmatrix}$$

> **Index legend:**
>
> | Row/Column | Index | What it labels | Physical meaning |
> |-----------|-------|----------------|-----------------|
> | **Rows** | $\alpha \in \{X, Y\}$ | **Antenna feed** | $X$ = the East–West oriented dipole feed on the antenna; $Y$ = the North–South oriented dipole feed. These are two physical hardware outputs — two voltage time-streams per antenna. |
> | **Columns** | $\gamma \in \{\hat{\theta}, \hat{\phi}\}$ | **Sky polarization** | $\hat{\theta}$ = the electric-field direction on the sky pointing along increasing zenith angle (roughly "up-down"); $\hat{\phi}$ = the electric-field direction pointing along increasing azimuth (roughly "left-right"). These describe the incoming wave, not the antenna. |
>
> | Element | Name | Meaning |
> |---------|------|---------|
> | $J_p^{X\hat{\theta}}$ | Co-polar ($X$-feed) | Response of the $X$-feed to $\hat{\theta}$-polarized radiation. This is typically the **dominant** response for the $X$-feed. |
> | $J_p^{X\hat{\phi}}$ | Cross-polar ($X$-feed) | Response of the $X$-feed to $\hat{\phi}$-polarized radiation. **Ideally small** — represents leakage. |
> | $J_p^{Y\hat{\theta}}$ | Cross-polar ($Y$-feed) | Response of the $Y$-feed to $\hat{\theta}$-polarized radiation. Ideally small. |
> | $J_p^{Y\hat{\phi}}$ | Co-polar ($Y$-feed) | Response of the $Y$-feed to $\hat{\phi}$-polarized radiation. Dominant response for $Y$-feed. |

**Physical picture:** An incoming wave from direction $\hat{\mathbf{s}}$ has an electric field with components $(E^{\hat{\theta}}, E^{\hat{\phi}})$. The two feeds on antenna $p$ produce voltages:

$$\begin{pmatrix} v_p^X \\ v_p^Y \end{pmatrix} = \begin{pmatrix} J_p^{X\hat{\theta}} & J_p^{X\hat{\phi}} \\ J_p^{Y\hat{\theta}} & J_p^{Y\hat{\phi}} \end{pmatrix} \begin{pmatrix} E^{\hat{\theta}} \\ E^{\hat{\phi}} \end{pmatrix}$$

For a perfect antenna with zero cross-polarization, the off-diagonal elements vanish ($J^{X\hat{\phi}} = J^{Y\hat{\theta}} = 0$) and the Jones matrix is diagonal.

The full $2 \times 2$ visibility matrix is:

$$\mathbf{V}_{pq} = \begin{pmatrix} V_{pq}^{XX} & V_{pq}^{XY} \\ V_{pq}^{YX} & V_{pq}^{YY} \end{pmatrix}$$

> **Index legend:**
>
> | Element | Correlation | Physical sensitivity |
> |---------|------------|---------------------|
> | $V_{pq}^{XX}$ | $\langle v_p^X \cdot v_q^{X*} \rangle$ | Primarily Stokes $I + Q$. |
> | $V_{pq}^{XY}$ | $\langle v_p^X \cdot v_q^{Y*} \rangle$ | Primarily Stokes $U + iV$. |
> | $V_{pq}^{YX}$ | $\langle v_p^Y \cdot v_q^{X*} \rangle$ | Primarily Stokes $U - iV$. |
> | $V_{pq}^{YY}$ | $\langle v_p^Y \cdot v_q^{Y*} \rangle$ | Primarily Stokes $I - Q$. |

The **four visibility products** $XX, XY, YX, YY$ together contain the full polarimetric information. Stokes parameters can be recovered: $I = (V^{XX} + V^{YY})/2$, $Q = (V^{XX} - V^{YY})/2$, etc. (in the ideal case).

### 8.2 Polarized Source Contribution

For a source with full Stokes coherency, each visibility element involves a **double sum** over sky polarizations:

$$V_{pq}^{\alpha\beta} = \sum_s \sum_{\gamma,\delta} J_p^{\alpha\gamma} \, C_s^{\gamma\delta} \, J_q^{\beta\delta *} \, e^{-2\pi i \nu \mathbf{b}_{pq} \cdot \hat{\mathbf{s}}_s / c}$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $\alpha, \beta$ | Antenna feed indices ($X$ or $Y$), selecting the feeds on antennas $p$ and $q$. |
> | $\gamma, \delta$ | Sky polarization indices ($\hat{\theta}$ or $\hat{\phi}$) being summed over. The sum has $2 \times 2 = 4$ terms: $(\hat{\theta}\hat{\theta}), (\hat{\theta}\hat{\phi}), (\hat{\phi}\hat{\theta}), (\hat{\phi}\hat{\phi})$. |
> | $s$ | Source index (summed over all sources). |
> | $C_s^{\gamma\delta}$ | Element $(\gamma, \delta)$ of the $2 \times 2$ coherency matrix of source $s$. For an unpolarized source, only $\gamma = \delta$ terms survive. |
> | $J_q^{\beta\delta *}$ | Complex conjugate of the Jones element of antenna $q$, feed $\beta$, sky polarization $\delta$. |

### 8.3 matvis Implementation for Polarization

For the fully polarized case, `matvis` constructs per-antenna vectors for each feed–sky polarization combination:

$$A_p^{\alpha\gamma}(s) = J_p^{\alpha\gamma}(\hat{\mathbf{s}}_s) \cdot \sqrt{C_s^{\gamma\delta}} \cdot e^{-2\pi i \nu \tau_{p,s}}$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $A_p^{\alpha\gamma}(s)$ | Per-antenna measurement element: antenna $p$, feed $\alpha$, coupling to sky polarization $\gamma$, from source $s$. |
> | $\alpha$ | Antenna feed index (which dipole: $X$ or $Y$). |
> | $\gamma$ | Sky polarization index (which sky component: $\hat{\theta}$ or $\hat{\phi}$). |

The visibility for a specific feed pair $(\alpha, \beta)$ accumulates contributions from each sky polarization pair:

$$V_{pq}^{\alpha\beta} = \sum_{\gamma,\delta} \left[\mathbf{A}^{\alpha\gamma} (\mathbf{A}^{\beta\delta})^H\right]_{pq}$$

> **Index legend:** The sum over $(\gamma, \delta)$ runs over $\{(\hat{\theta},\hat{\theta}), (\hat{\theta},\hat{\phi}), (\hat{\phi},\hat{\theta}), (\hat{\phi},\hat{\phi})\}$ — i.e., all 4 sky polarization pairs. Each term requires one matrix multiplication. In the Stokes I case, the sum collapses to 1 or 2 terms.

### 8.4 Feed Types and Configurations

`matvis` supports different feed configurations:

- **Single polarization (1 feed):** Only one feed ($X$ or $Y$) per antenna is used. The Jones "matrix" is a $1 \times 2$ row vector: $\mathbf{J}_p = (J_p^{X\hat{\theta}}, J_p^{X\hat{\phi}})$. Only one visibility product is computed (e.g., $V^{XX}$). Use this when only Stokes $I$ is needed and cross-polarization can be ignored.

- **Dual polarization (2 feeds):** Both $X$ and $Y$ feeds are used, giving the full $2 \times 2$ Jones matrix. All four visibility products ($XX, XY, YX, YY$) are computed. This is needed for full polarimetry.

- **`polarized` flag:** This boolean parameter controls the mode:
  - `polarized=False` (default): Uses a **scalar beam** — a single complex number per (antenna, direction, frequency). The coherency is just $I_s$. One matrix multiply per frequency per time. Fast. Appropriate for unpolarized sky models.
  - `polarized=True`: Uses the full $2 \times 2$ Jones matrix and full Stokes IQUV coherency. Four (or more) matrix multiplies per frequency per time. Required for polarimetric accuracy.

### 8.5 Stokes I Only Simplification

When `polarized=False`:
- Only one (scalar) beam component per feed is used, ignoring cross-polarization
- The coherency reduces to just $I_s$ (a scalar)
- One matrix multiply per frequency per time (instead of up to 4)
- This is $\sim 4\times$ faster than the fully polarized case

**When is this a good approximation?**
1. Sources are known to be unpolarized (or polarization fraction $\ll 1$)
2. The antenna cross-polarization is small ($J^{X\hat{\phi}}, J^{Y\hat{\theta}} \approx 0$)
3. Only Stokes $I$ visibilities are needed

For 21 cm cosmology (e.g., HERA), foreground sources are typically weakly polarized ($\lesssim$ few percent) and the signal of interest is Stokes $I$, so the scalar beam approximation is usually adequate.

In [ ]:
# Search for polarization handling code
for f in sorted(all_py_files):
    with open(f, 'r') as fh:
        content = fh.read()
    
    if 'polarized' in content or 'nfeed' in content or 'jones' in content.lower():
        relpath = os.path.relpath(f, matvis_path)
        lines = content.split('\n')
        for i, line in enumerate(lines):
            if any(kw in line.lower() for kw in ['polarized', 'nfeed', 'n_feed', 'jones', 'stokes']):
                if not line.strip().startswith('#') and len(line.strip()) > 3:
                    print(f"{relpath}:{i+1}: {line.strip()[:120]}")

---
## 9. Frequency & Time Looping Structure <a id='9-loops'></a>

### 9.1 Overall Loop Structure

The pseudocode below shows how `matvis` organises the computation. Every variable is annotated with its shape and the indices it carries.

```
for t in range(N_times):                           # Outer loop: TIME index t
    crd_top = eq2tops[t] @ crd_eq                  # Coordinate rotation (once per time)
                                                    # shape: (3, N_src) — topocentric (l,m,n) for all sources
    above_horizon = crd_top[2] > 0                 # Horizon mask — boolean array, shape (N_src,)
    
    # Compute az/za for beam evaluation
    az, za = compute_angles(crd_top[:, above_horizon])  # az, za each shape (N_src_above,)
    
    # Compute geometric delay (once per time, frequency-independent)
    tau = antpos @ crd_top[:, above_horizon] / c   # shape (N_ant, N_src_above) — delay in seconds
    
    for k in range(N_freq):                        # Inner loop: FREQUENCY index k
        # Evaluate beam at this frequency
        beam_vals = evaluate_beam(beam, az, za, freq[k])  # shape (N_ant, N_src_above) [scalar]
                                                           # or (N_feed, N_ant, N_src_above) [polarized]
        
        # Normalize beam by zenith value
        beam_vals /= beam_at_zenith[k]             # Peak normalization (see Section 4)
        
        # Compute fringe
        fringe = exp(-2j * pi * freq[k] * tau)     # shape (N_ant, N_src_above) — complex phasor
        
        # Build per-antenna vector
        A = beam_vals * sqrt(I_sky[k, above_horizon]) * fringe  # shape (N_ant, N_src_above)
        
        # Compute all visibilities via matrix multiply
        V[:, :, k, t] = A @ A.conj().T             # shape (N_ant, N_ant) — the outer product
```

> **Index legend for the pseudocode:**
>
> | Variable | Index / Shape | Meaning |
> |----------|--------------|---------|
> | `t` | Scalar, $t \in \{0, \ldots, N_{\text{time}}-1\}$ | Time step index. Each value corresponds to one LST snapshot. |
> | `k` | Scalar, $k \in \{0, \ldots, N_{\text{freq}}-1\}$ | Frequency channel index. |
> | `eq2tops[t]` | $(3 \times 3)$ matrix | Rotation matrix converting equatorial direction cosines to topocentric ENU at time $t$. |
> | `crd_eq` | $(3, N_{\text{src}})$ | Equatorial source direction cosines. Fixed for the entire observation (sky is inertial). |
> | `crd_top` | $(3, N_{\text{src}})$ | Topocentric source direction cosines at time $t$. Row 0 = $l$ (East), Row 1 = $m$ (North), Row 2 = $n$ (Up). |
> | `above_horizon` | $(N_{\text{src}},)$ boolean | `True` for sources with $n > 0$ (above the horizon); `False` otherwise. |
> | `az, za` | $(N_{\text{src\_above}},)$ each | Azimuth and zenith angle of each above-horizon source. $\text{za} = \arccos(n)$, $\text{az} = \arctan2(l, m)$. |
> | `tau` | $(N_{\text{ant}}, N_{\text{src\_above}})$ | Geometric delay $\tau_{p,s} = \hat{\mathbf{s}}_s \cdot \mathbf{r}_p / c$ in seconds. **Frequency-independent.** |
> | `freq[k]` | Scalar | Frequency of channel $k$, in Hz. |
> | `beam_vals` | $(N_{\text{ant}}, N_{\text{src\_above}})$ | Beam response (amplitude & phase) at each source direction for each antenna, at frequency $\nu_k$. |
> | `beam_at_zenith[k]` | Scalar (or per-antenna) | Beam value at the zenith direction for normalization. |
> | `fringe` | $(N_{\text{ant}}, N_{\text{src\_above}})$ | The complex phasor $e^{-2\pi i \nu_k \tau_{p,s}}$. |
> | `I_sky[k, :]` | $(N_{\text{src}},)$ | Source flux densities at frequency $\nu_k$, in Jy. |
> | `A` | $(N_{\text{ant}}, N_{\text{src\_above}})$ | The per-antenna measurement vector (see Section 7). |
> | `V[:, :, k, t]` | $(N_{\text{ant}}, N_{\text{ant}})$ | Visibility matrix — all baselines at once for time $t$, frequency $\nu_k$. |

### 9.2 Key Optimization: Time-Frequency Separation

**Critical numerical choice:** The geometric delay $\tau_{p,s}$ depends only on the source position and antenna position — it is **frequency-independent**:

$$\tau_{p,s} = \frac{\hat{\mathbf{s}}_s \cdot \mathbf{r}_p}{c}$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $\tau_{p,s}$ | Geometric delay for antenna $p$ and source $s$, in seconds. |
> | $\hat{\mathbf{s}}_s$ | Unit direction vector to source $s$ in topocentric coordinates. Changes with time $t$ but not frequency. |
> | $\mathbf{r}_p$ | Position of antenna $p$ in meters (ENU). Fixed. |
> | $c$ | Speed of light, $2.998 \times 10^8$ m/s. |

Therefore:

1. $\tau$ is computed **once per time step** (in the outer `t` loop, outside the `k` loop)
2. The exponential $e^{-2\pi i \nu_k \tau}$ is computed for each frequency in the inner loop — this is a cheap element-wise multiply
3. This avoids redundantly recomputing the dot product $\hat{\mathbf{s}} \cdot \mathbf{r}$ for every frequency

### 9.3 Source Filtering Per Time Step (Horizon Cut)

At each time step, only sources above the horizon contribute:

$$S_{\text{above}}(t) = \{s : n_s(t) > 0\}$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $S_{\text{above}}(t)$ | The set of source indices that are above the horizon at time $t$. |
> | $n_s(t)$ | The third topocentric coordinate (Up component) of source $s$ at time $t$. Equals $\cos(\text{zenith angle})$. Positive means above horizon. |

**Numerical savings:** For a full-sky catalog, roughly half the sources are below the horizon at any time. The horizon cut therefore approximately halves the number of sources, halving the cost of all subsequent operations (beam evaluation, fringe computation, matrix multiplication).

**Threshold choice:** The standard threshold is $n > 0$ (exactly at the geometric horizon). Some implementations use a small positive threshold $n > \epsilon$ to avoid sources at the horizon where the beam may be poorly defined.

### 9.4 Memory Layout

The output visibility array is typically stored as:
- **Shape:** $(N_{\text{time}}, N_{\text{freq}}, N_{\text{ant}}, N_{\text{ant}})$ or a flattened baseline-indexed version
- **Data type:** Complex128 (double precision complex) by default — 16 bytes per visibility
- **Upper triangle:** The matrix $\mathbf{V} = \mathbf{A}\mathbf{A}^H$ is Hermitian, so $V_{pq} = V_{qp}^*$. Only the upper triangle (with $p \leq q$) contains independent information.
- **Diagonal:** $V_{pp}$ is the auto-correlation of antenna $p$ — real and non-negative. $V_{pp} = \sum_s |A_p(s)|^2$.

**Memory estimate:** For HERA-350 with 1024 frequency channels and 100 time steps:
- Full matrix: $100 \times 1024 \times 350 \times 350 \times 16$ bytes $\approx$ **200 GB**
- This is why visibilities are typically written to disk per time step or accumulated in a streaming fashion.

In [ ]:
# Print the full core simulation function to see the actual loop structure
for f in sorted(all_py_files):
    with open(f, 'r') as fh:
        content = fh.read()
    
    relpath = os.path.relpath(f, matvis_path)
    
    # Look for the main simulation function
    if ('def simulate' in content or 'def _simulate' in content or 
        'def vis_cpu' in content or 'def vis_loop' in content or
        'for freq' in content.lower()):
        
        # Check if this file contains the actual compute loop
        if 'eq2top' in content or 'crd_eq' in content:
            print(f"\n{'='*80}")
            print(f"MAIN SIMULATION CODE: {relpath}")
            print(f"{'='*80}")
            print(content)
            print()

---
## 10. GPU vs CPU Implementation Differences <a id='10-gpu-cpu'></a>

### 10.1 CPU Implementation

The CPU implementation uses:
- **NumPy** for array operations (element-wise multiplications, array slicing)
- **BLAS** (via NumPy/SciPy) for the matrix multiplication `A @ A.conj().T` — this is typically handled by OpenBLAS or Intel MKL behind the scenes
- Python-level `for` loops over time ($t$) and frequency ($k$)
- The beam evaluation calls Python/NumPy functions (e.g., `UVBeam.interp()` or analytic beam functions)

**Performance characteristics:**
- The matrix multiply $\mathbf{A} \mathbf{A}^H$ is the dominant cost and is handled by highly optimized BLAS routines
- For large $N_{\text{ant}}$ (e.g., HERA-350), BLAS can use multiple CPU cores
- The Python loop overhead is small compared to the BLAS time

### 10.2 GPU Implementation

The GPU implementation uses:
- **CuPy** or **PyCUDA** for array operations on NVIDIA GPUs
- **cuBLAS** for the matrix multiplication $\mathbf{A}\mathbf{A}^H$ — the GPU equivalent of BLAS
- Custom CUDA kernels for beam interpolation and fringe computation
- Data is transferred to GPU memory once, and the entire frequency loop runs on the GPU without transferring data back

**Performance characteristics:**
- GPU excels when $N_{\text{ant}}$ and $N_{\text{src}}$ are large (thousands of sources, hundreds of antennas)
- The matrix multiplication is particularly well-suited to GPU parallelism
- Data transfer (CPU → GPU) is a one-time cost at the start of each time step

### 10.3 Mathematical Equivalence

Both implementations compute the **same mathematical operation**:

$$V_{pq}^{\alpha\beta}(\nu_k, t) = \sum_{s \in S_{\text{above}}(t)} I_s(\nu_k) \; \tilde{J}_p^{\alpha}(\hat{\mathbf{s}}_s, \nu_k) \; \tilde{J}_q^{\beta *}(\hat{\mathbf{s}}_s, \nu_k) \; e^{-2\pi i \nu_k \mathbf{b}_{pq} \cdot \hat{\mathbf{s}}_s / c}$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $V_{pq}^{\alpha\beta}(\nu_k, t)$ | Visibility for baseline $(p, q)$, feed pair $(\alpha, \beta)$, at frequency $\nu_k$ and time $t$. |
> | $p, q$ | Antenna indices. |
> | $\alpha, \beta$ | Feed indices ($X$ or $Y$) on antennas $p$ and $q$. |
> | $\nu_k$ | Frequency of channel $k$. |
> | $t$ | Time step index. |
> | $s$ | Source index, summed over all sources in $S_{\text{above}}(t)$ — those above the horizon at time $t$. |
> | $I_s(\nu_k)$ | Flux density of source $s$ at frequency $\nu_k$, in Jy. |
> | $\tilde{J}_p^{\alpha}$ | Peak-normalized beam response of antenna $p$, feed $\alpha$, in the direction of source $s$ at frequency $\nu_k$. Dimensionless. |
> | $\tilde{J}_q^{\beta *}$ | Complex conjugate of the peak-normalized beam of antenna $q$, feed $\beta$. The conjugation arises from the correlation operation. |
> | $\mathbf{b}_{pq}$ | Baseline vector $\mathbf{r}_p - \mathbf{r}_q$, in meters. |
> | $\hat{\mathbf{s}}_s$ | Unit direction vector toward source $s$ in topocentric ENU coordinates. |
> | $c$ | Speed of light (m/s). |

### 10.4 Numerical Differences Between CPU and GPU

Although mathematically identical, there are small numerical differences:

| Aspect | CPU (NumPy/BLAS) | GPU (CuPy/cuBLAS) |
|--------|-------------------|---------------------|
| Default precision | float64 / complex128 | Can be float32 / complex64 |
| Summation order | Deterministic (fixed BLAS algorithm) | May vary (GPU parallelism can reorder additions) |
| Typical agreement | — | $\sim 10^{-12}$ (float64) or $\sim 10^{-5}$ (float32) |

These differences are of order machine epsilon and are scientifically negligible. However, if exact bitwise reproducibility is required, use the CPU backend with a deterministic BLAS.

In [ ]:
# Check for GPU implementation
gpu_files = glob.glob(os.path.join(matvis_path, '**/gpu*.py'), recursive=True)
gpu_files += glob.glob(os.path.join(matvis_path, '**/cuda*.py'), recursive=True)

if gpu_files:
    for f in sorted(gpu_files):
        relpath = os.path.relpath(f, matvis_path)
        print(f"GPU file found: {relpath}")
        with open(f, 'r') as fh:
            content = fh.read()
        print(content[:3000])
else:
    print("No dedicated GPU files found. GPU support may be integrated into main code or via a separate package.")

---
## 11. Summary of All Numerical Choices <a id='11-summary'></a>

### Complete Step-by-Step Algorithm

| Step | Operation | Math | Index Legend | Reason |
|------|-----------|------|-------------|--------|
| 1 | **Coordinate transform** | $\hat{\mathbf{s}}_{\text{top}} = \mathbf{R}(t) \cdot \hat{\mathbf{s}}_{\text{eq}}$ | $\mathbf{R}(t)$: $3\times3$ rotation matrix at time $t$; $\hat{\mathbf{s}}_{\text{eq}}$: equatorial direction cosines of a source; $\hat{\mathbf{s}}_{\text{top}}$: topocentric (ENU) direction cosines | Convert source positions from equatorial to topocentric frame at each LST |
| 2 | **Horizon cut** | Keep $s$ if $n_s > 0$ | $n_s$: the Up-component (3rd coordinate) of source $s$ in topocentric frame | Sources below horizon do not contribute; saves roughly 50% computation for full-sky catalogs |
| 3 | **Compute angles** | $\theta_s = \arccos(n_s)$, $\phi_s = \arctan2(l_s, m_s)$ | $\theta_s$: zenith angle of source $s$; $\phi_s$: azimuth angle; $l_s, m_s, n_s$: topocentric direction cosines (East, North, Up) | Beam models expect (zenith angle, azimuth) as input coordinates |
| 4 | **Geometric delay** | $\tau_{p,s} = \hat{\mathbf{s}}_s \cdot \mathbf{r}_p / c$ | $\tau_{p,s}$: delay in seconds for antenna $p$, source $s$; $\mathbf{r}_p$: antenna position (m); $c$: speed of light | Frequency-independent; computed once per time step $t$ and reused across all $N_{\text{freq}}$ channels |
| 5 | **Beam evaluation** | $J_p^{\alpha\gamma}(\theta_s, \phi_s, \nu_k)$ | $\alpha$: feed index ($X$/$Y$); $\gamma$: sky pol ($\hat\theta$/$\hat\phi$); $\nu_k$: frequency of channel $k$ | Interpolate or analytically compute beam response at source positions |
| 6 | **Beam normalization** | $\tilde{J} = J / J(\hat{\mathbf{z}})$ | $\hat{\mathbf{z}}$: zenith direction ($\theta=0$); $\tilde{J}$: normalized beam (dimensionless) | **Peak normalization** at zenith; ensures visibilities are in Jy |
| 7 | **Fringe phasor** | $e^{-2\pi i \nu_k \tau_{p,s}}$ | $\nu_k$: channel frequency; $\tau_{p,s}$: per-antenna delay | Per-antenna phase; baseline phase $e^{-2\pi i \nu \mathbf{b}_{pq}\cdot\hat{\mathbf{s}}/c}$ emerges from outer product |
| 8 | **Source weighting** | $\sqrt{I_s(\nu_k)}$ | $I_s(\nu_k)$: flux density of source $s$ at frequency $\nu_k$ (Jy) | Square root because $\sqrt{I} \times \sqrt{I} = I$ is recovered in the outer product |
| 9 | **Per-antenna vector** | $A_p^\alpha(s) = \tilde{J}_p^\alpha \cdot \sqrt{I_s} \cdot e^{-2\pi i \nu_k \tau_{p,s}}$ | $A_p^\alpha(s)$: entry of the per-antenna matrix for antenna $p$, feed $\alpha$, source $s$ | Combines all per-(antenna, source) factors into one array |
| 10 | **Matrix multiply** | $\mathbf{V}^{\alpha\beta} = \mathbf{A}^\alpha (\mathbf{A}^\beta)^H$ | $\mathbf{A}^\alpha$: shape $(N_{\text{ant}}, N_{\text{src}})$; superscript $H$: conjugate transpose | Computes all baselines simultaneously; $O(N_{\text{ant}}^2 N_{\text{src}})$ via BLAS |
| 11 | **Extract baselines** | $V_{pq}$ from matrix element $(p, q)$ | $p, q$: antenna indices identifying a unique baseline | Upper triangle ($p < q$): cross-correlations; diagonal ($p = q$): auto-correlations |

### Detailed Numerical Choices and Justifications

#### Choice 1: Point Source Discretization (No $d\Omega$ factor)
- **What:** The integral $\int d\Omega$ is replaced by a discrete sum $\sum_s$. No pixel solid-angle element appears.
- **Why:** For a point-source sky model, each source has a well-defined total flux density $I_s$ in Jy. The $d\Omega$ would only be needed for continuous (diffuse) emission, where the sky brightness is in Jy/sr and must be multiplied by a pixel area.
- **Implication:** Cannot simulate continuous diffuse emission directly. To include diffuse emission, one would pixelize the sky into many small patches, each treated as a point source with flux $= B(\hat{\mathbf{s}}) \, \Delta\Omega$, where $B$ is the surface brightness (Jy/sr) and $\Delta\Omega$ is the pixel solid angle.

> **Index legend:** $s$ = source index ($1 \leq s \leq N_{\text{src}}$); $I_s$ = flux density of source $s$ in Jy; $\Delta\Omega$ = solid angle of one sky pixel (sr).

#### Choice 2: Per-Antenna Factorization ($\mathbf{A}\mathbf{A}^H$ formulation)
- **What:** Instead of looping over $N_{\text{bl}} = N_{\text{ant}}(N_{\text{ant}}-1)/2$ baselines, construct $\mathbf{A} \in \mathbb{C}^{N_{\text{ant}} \times N_{\text{src}}}$ and compute $\mathbf{V} = \mathbf{A}\mathbf{A}^H$.
- **Why:** Matrix multiplication is highly optimized in BLAS/cuBLAS. For HERA-350, $N_{\text{bl}} \approx 61{,}000$. A baseline-by-baseline loop would make 61,000 passes through the source list. The matrix approach computes everything in one BLAS call.
- **Complexity:** $O(N_{\text{ant}}^2 \times N_{\text{src}})$ floating-point operations per frequency per time step.
- **Implication:** Computes ALL baselines including auto-correlations and both orientations $(p,q)$ and $(q,p)$. Memory footprint is dominated by $\mathbf{A}$: $N_{\text{ant}} \times N_{\text{src}} \times 16$ bytes (complex128).

> **Index legend:** $\mathbf{A} \in \mathbb{C}^{N_{\text{ant}} \times N_{\text{src}}}$ — rows indexed by antenna $p$, columns by source $s$; $\mathbf{V} \in \mathbb{C}^{N_{\text{ant}} \times N_{\text{ant}}}$ — rows and columns indexed by antennas $p, q$.

#### Choice 3: $\sqrt{I_s}$ Factorization
- **What:** The flux density is split as $I_s = \sqrt{I_s} \times \sqrt{I_s}$ so that each antenna carries a factor of $\sqrt{I_s}$.
- **Why:** Enables the per-antenna factorization. Without this trick, the flux $I_s$ would be a per-baseline quantity that cannot be absorbed into $\mathbf{A}$.
- **Implication:** Only works for $I_s \geq 0$. For negative flux sources (rare, but possible in difference maps or residual images), $\sqrt{I_s}$ is imaginary, and the formulation would need modification (e.g., tracking sign separately).

> **Index legend:** $I_s$ = Stokes I flux density of source $s$ (Jy); $\sqrt{I_s}$ = factor absorbed into each antenna's measurement vector $A_p(s)$.

#### Choice 4: Peak Beam Normalization
- **What:** $\tilde{J}(\hat{\mathbf{s}}) = J(\hat{\mathbf{s}}) / J(\hat{\mathbf{z}})$ — divide by the zenith beam value.
- **Why:** Standard radio astronomy convention. A 1 Jy point source at zenith should produce exactly 1 Jy of correlated flux in the visibility.
- **Contrast with area normalization:** An area-normalized beam would satisfy $\int |\tilde{J}|^2 \, d\Omega = 1$, which embeds the beam solid angle $\Omega_B$ into the normalization. Peak normalization does NOT include $\Omega_B$; it must be handled separately when converting to brightness temperature:
$$T_b = \frac{\lambda^2}{2 k_B \Omega_B} S$$

> **Index legend:** $\hat{\mathbf{z}}$ = zenith direction; $\Omega_B = \int |J/J_{\max}|^2 d\Omega$ = beam solid angle (sr); $\lambda$ = wavelength (m); $k_B$ = Boltzmann constant (J/K); $S$ = flux density (W m$^{-2}$ Hz$^{-1}$).

#### Choice 5: Horizon Cut at $n > 0$
- **What:** Sources with topocentric $z$-component $n_s \leq 0$ are excluded from the sum.
- **Why:** Below-horizon sources are physically unobservable — the ground blocks them. Including them would add spurious signal.
- **Implication:** Sources very close to the horizon ($n_s \approx 0$) may have poorly defined beam response (the beam pattern near $90°$ zenith angle is often unreliable). Some simulators use a slightly positive threshold $n_s > \epsilon$ (e.g., $\epsilon = 0.01$, corresponding to zenith angle $> 89.4°$).

> **Index legend:** $n_s(t)$ = the Up-component of source $s$ in topocentric coordinates at time $t$; equals $\cos(\theta_s)$ where $\theta_s$ is zenith angle.

#### Choice 6: Separate Geometric Delay from Frequency
- **What:** Compute $\tau = \hat{\mathbf{s}} \cdot \mathbf{r} / c$ once per time step, then $e^{-2\pi i \nu_k \tau}$ per frequency.
- **Why:** The dot product $\hat{\mathbf{s}} \cdot \mathbf{r}$ and the horizon cut are the expensive parts; computing $e^{i\phi}$ from a stored $\tau$ is cheap.
- **Implication:** Assumes non-dispersive propagation — the geometric delay $\tau$ is achromatic (the same at all frequencies). This breaks down if ionospheric refraction is significant (not modeled by matvis).

> **Index legend:** $\tau_{p,s}$ = geometric delay (s) for antenna $p$, source $s$; $\nu_k$ = frequency of channel $k$ (Hz).

#### Choice 7: Complex Double Precision (complex128)
- **What:** Default computation uses float64 / complex128 (16 bytes per complex number).
- **Why:** Radio interferometric visibilities require high dynamic range. At long baselines and high frequencies, the fringe phase $2\pi\nu\tau$ can be very large ($\sim 10^{10}$ radians), and float32's $\sim 7$ significant digits would introduce phase errors of order $10^3$ radians.
- **Implication:** 2× memory and $\sim$ 2× computation cost compared to float32. For GPU computation, float32 may be used if the baseline lengths and frequencies are modest enough that phase precision is acceptable.

---
## Appendix A: Complete Mathematical Summary

This appendix collects all the key equations in one place for quick reference.

### A.1 The Continuous RIME (what we want to simulate)

$$\boxed{\mathbf{V}_{pq}(\nu) = \int_{4\pi} \mathbf{J}_p(\hat{\mathbf{s}}, \nu) \; \mathbf{C}(\hat{\mathbf{s}}, \nu) \; \mathbf{J}_q^H(\hat{\mathbf{s}}, \nu) \; e^{-2\pi i \nu \, \mathbf{b}_{pq} \cdot \hat{\mathbf{s}} / c} \; d\Omega}$$

> **Index legend:** $p, q$ = antenna indices; $\nu$ = frequency; $\hat{\mathbf{s}}$ = sky direction (integration variable); $\mathbf{J}_p$ = $2\times2$ Jones matrix of antenna $p$; $\mathbf{C}$ = $2\times2$ coherency matrix of the sky; $\mathbf{b}_{pq} = \mathbf{r}_p - \mathbf{r}_q$ = baseline vector; $c$ = speed of light; $d\Omega$ = solid angle element. See Section 1.1 for details.

### A.2 The Discrete RIME (point source approximation)

$$\boxed{\mathbf{V}_{pq}(\nu_k) = \sum_{s=1}^{N_{\text{src}}} \mathbf{J}_p(\hat{\mathbf{s}}_s, \nu_k) \; \mathbf{C}_s(\nu_k) \; \mathbf{J}_q^H(\hat{\mathbf{s}}_s, \nu_k) \; e^{-2\pi i \nu_k \, \mathbf{b}_{pq} \cdot \hat{\mathbf{s}}_s / c}}$$

> **Index legend:** $s$ = source index ($1 \leq s \leq N_{\text{src}}$); $\nu_k$ = frequency of channel $k$; $\hat{\mathbf{s}}_s$ = unit direction to source $s$; $\mathbf{C}_s$ = coherency matrix of source $s$ (in Jy, not Jy/sr). All other symbols same as A.1. See Section 1.2.

### A.3 The matvis Matrix Formulation (Stokes I only)

**Per-antenna vector:**
$$\boxed{A_p^{\alpha}(s, \nu_k) = \frac{J_p^{\alpha}(\hat{\mathbf{s}}_s, \nu_k)}{J_p^{\alpha}(\hat{\mathbf{z}}, \nu_k)} \cdot \sqrt{I_s(\nu_k)} \cdot \exp\left(-\frac{2\pi i \nu_k}{c} \hat{\mathbf{s}}_s \cdot \mathbf{r}_p\right)}$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $A_p^{\alpha}(s, \nu_k)$ | Element of the per-antenna measurement vector — antenna $p$, feed $\alpha$, source $s$, frequency $\nu_k$. |
> | $J_p^{\alpha}(\hat{\mathbf{s}}_s, \nu_k)$ | Beam of antenna $p$, feed $\alpha$, evaluated at source direction $\hat{\mathbf{s}}_s$ and frequency $\nu_k$. |
> | $J_p^{\alpha}(\hat{\mathbf{z}}, \nu_k)$ | Beam of antenna $p$, feed $\alpha$, evaluated at **zenith** — used for peak normalization. |
> | $\sqrt{I_s(\nu_k)}$ | Square root of source flux density — each antenna carries this factor so the product recovers $I_s$. |
> | $\hat{\mathbf{s}}_s \cdot \mathbf{r}_p$ | Dot product of source direction with antenna position — the geometric path length (meters). Divided by $c$ to get delay. |

**Visibility via outer product:**
$$\boxed{V_{pq}^{\alpha\beta}(\nu_k) = \sum_{s \in S_{\text{above}}} A_p^{\alpha}(s, \nu_k) \; A_q^{\beta *}(s, \nu_k) = \left[\mathbf{A}^{\alpha} (\mathbf{A}^{\beta})^H\right]_{pq}}$$

> **Index legend:** $p, q$ = antenna indices; $\alpha, \beta$ = feed indices ($X$ or $Y$); $s$ = source index summed over above-horizon sources $S_{\text{above}}$; $\mathbf{A}^{\alpha}$ is the matrix of shape $(N_{\text{ant}}, N_{\text{src}})$ with entries $A_p^{\alpha}(s)$; $(\mathbf{A}^{\beta})^H$ = its conjugate transpose (shape $N_{\text{src}} \times N_{\text{ant}}$). The product is shape $(N_{\text{ant}} \times N_{\text{ant}})$.

### A.4 Equivalence Proof

Expanding the outer product element $(p, q)$:

$$\sum_s A_p^\alpha A_q^{\beta*} = \sum_s \underbrace{\frac{J_p^\alpha(s)}{J_p^\alpha(z)}}_{\tilde{J}_p^\alpha} \underbrace{\sqrt{I_s}}_{\text{source}} \, \underbrace{e^{-2\pi i \nu \hat{s} \cdot r_p/c}}_{\text{fringe}_p} \cdot \underbrace{\frac{J_q^{\beta*}(s)}{J_q^{\beta*}(z)}}_{\tilde{J}_q^{\beta*}} \underbrace{\sqrt{I_s}}_{\text{source}} \, \underbrace{e^{+2\pi i \nu \hat{s} \cdot r_q/c}}_{\text{fringe}_q^*}$$

Combining the two $\sqrt{I_s}$ factors and the two exponentials:

$$= \sum_s \tilde{J}_p^\alpha(s) \, I_s \, \tilde{J}_q^{\beta*}(s) \, e^{-2\pi i \nu (\hat{s} \cdot r_p - \hat{s} \cdot r_q)/c}$$

Using $\hat{\mathbf{s}} \cdot \mathbf{r}_p - \hat{\mathbf{s}} \cdot \mathbf{r}_q = \hat{\mathbf{s}} \cdot (\mathbf{r}_p - \mathbf{r}_q) = \hat{\mathbf{s}} \cdot \mathbf{b}_{pq}$:

$$= \sum_s \tilde{J}_p^\alpha(s) \, I_s \, \tilde{J}_q^{\beta*}(s) \, e^{-2\pi i \nu \, \mathbf{b}_{pq} \cdot \hat{\mathbf{s}}/c} \quad \checkmark$$

> This is exactly the discrete RIME (A.2) for Stokes I with peak-normalized beams, confirming the equivalence. $\square$

---
## Appendix B: Beam Details Deep Dive

### B.1 UVBeam Interpolation Details

When using `UVBeam` objects from `pyuvdata`:

1. The beam data is stored on a regular grid in $(\theta, \phi)$ at discrete frequencies $\{\nu_1, \nu_2, \ldots\}$
2. Interpolation to the requested source positions $(\theta_s, \phi_s, \nu_k)$ uses the `UVBeam.interp()` method
3. Default interpolation is **bilinear** in angle and **cubic spline** in frequency
4. The beam can be stored in "efield" (Jones matrix) or "power" basis

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $\theta$ | Zenith angle coordinate on the beam grid (radians). $\theta = 0$ at zenith, $\pi/2$ at horizon. |
> | $\phi$ | Azimuth angle coordinate on the beam grid (radians). Measured East from North. |
> | $\theta_s, \phi_s$ | Zenith angle and azimuth of source $s$ — the point at which the beam must be interpolated. |
> | $\nu_k$ | Frequency at which the beam is evaluated. |

**matvis expects E-field (Jones) beam.** The relationship between the E-field beam and the power beam is:

$$P(\theta, \phi) = |J(\theta, \phi)|^2$$

> **Index legend:** $J(\theta, \phi)$ = electric-field (voltage) beam pattern (complex); $P(\theta, \phi)$ = power beam pattern (real, non-negative). The power beam is the square magnitude of the E-field beam. If $J$ is a $2\times2$ Jones matrix, then $P^{\alpha\gamma} = |J^{\alpha\gamma}|^2$.

### B.2 Beam Coordinate Convention

The beam is defined in a coordinate system aligned with the antenna/station. For an **alt-az** mount (typical for ground-based dipoles like HERA):
- The beam pattern is **fixed** relative to the ground — it does not rotate
- The sky rotates through the beam pattern as the Earth turns
- $\theta$ = zenith angle: $0$ at zenith, $\pi/2$ at the horizon
- $\phi$ = azimuth: measured East from North, typically $0°$ = North, $90°$ = East

This is why the coordinate transform (Section 2) converts source positions from equatorial to topocentric: the beam lives in the topocentric (ground-fixed) frame.

### B.3 Beam Spline Interpolation (if available)

Some versions of matvis support pre-computing beam splines for faster evaluation:

$$J^{\alpha\gamma}(\theta, \phi, \nu) \approx \sum_{i,j,k} c_{ijk} \, B_i(\theta) \, B_j(\phi) \, B_k(\nu)$$

> **Index legend:**
>
> | Symbol | Meaning |
> |--------|---------|
> | $c_{ijk}$ | Spline coefficients — pre-computed from the beam data grid. |
> | $B_i(\theta)$ | The $i$-th B-spline basis function in zenith angle. |
> | $B_j(\phi)$ | The $j$-th B-spline basis function in azimuth. |
> | $B_k(\nu)$ | The $k$-th B-spline basis function in frequency. |
> | $i, j, k$ | Indices running over the spline knots in each dimension. |
> | $\alpha$ | Feed index ($X$ or $Y$). |
> | $\gamma$ | Sky polarization index ($\hat{\theta}$ or $\hat{\phi}$). |

This tensor-product spline representation allows very fast evaluation at arbitrary $(\theta, \phi, \nu)$ triples once the coefficients $c_{ijk}$ have been computed.

### B.4 Per-Antenna vs Shared Beam

- **Shared beam:** All antennas use the same beam pattern: $J_p = J$ for all $p$. This is the most common case for identical antennas in a regular array like HERA. The beam is evaluated **once** per (direction, frequency) and the result is broadcast to all antennas.

- **Per-antenna beam:** Each antenna has its own beam pattern $J_p(\hat{\mathbf{s}}, \nu)$. This allows modeling:
  - Manufacturing variations between antenna elements
  - Different antenna types in a heterogeneous array
  - Position-dependent ground reflections (if each antenna sees a different ground screen)

When beams are shared, `matvis` evaluates the beam **once** per time-frequency step and applies it to all antennas, saving a factor of $N_{\text{ant}}$ in beam evaluation cost. This is typically the dominant saving, since beam interpolation can be expensive.

### B.5 Analytic Beam Models

For testing and quick simulations, `matvis` supports analytic beam models that avoid interpolation altogether:

- **Gaussian beam:** $J(\theta) = \exp\left(-\theta^2 / (2\sigma^2)\right)$, where $\sigma$ is the beam width parameter (radians). Azimuthally symmetric. Frequency dependence can be included via $\sigma(\nu) = \sigma_0 \cdot (\nu_0 / \nu)$ for a diffraction-limited aperture.

- **Airy beam:** $J(\theta) = 2 J_1(x) / x$ where $x = \pi D \sin(\theta) / \lambda$, $D$ is the dish diameter, $\lambda = c/\nu$ is the wavelength, and $J_1$ is the Bessel function of the first kind. This is the exact diffraction pattern of a circular aperture.

- **Uniform (isotropic):** $J(\theta, \phi) = 1$ everywhere. Used for testing only.

> **Index legend:** $\theta$ = zenith angle; $\sigma$ = Gaussian width (rad); $D$ = aperture diameter (m); $\lambda$ = wavelength (m); $J_1$ = first-order Bessel function (not to be confused with the Jones matrix).

In [ ]:
# Final: Print complete source of the main simulation engine
# to verify all the above documentation

# Find all Python files and print those that contain the core compute
for f in sorted(all_py_files):
    with open(f, 'r') as fh:
        content = fh.read()
    
    relpath = os.path.relpath(f, matvis_path)
    
    # The core simulation has both eq2tops and the matrix multiply
    if ('eq2top' in content and ('matmul' in content or '@' in content or '.dot' in content)):
        print(f"\n{'#'*80}")
        print(f"# CORE ENGINE: {relpath}")
        print(f"{'#'*80}")
        print(content)

---
## Appendix C: Connection to Physical Quantities

### C.1 Units

| Quantity | Symbol | Units in matvis | Notes |
|----------|--------|------------------|-------|
| Source flux density | $I_s$ | Jy ($10^{-26}$ W m$^{-2}$ Hz$^{-1}$) | Input per source |
| Antenna position | $\mathbf{r}_p$ | meters (ENU) | Topocentric East-North-Up |
| Frequency | $\nu$ (or $\nu_k$) | Hz | Per channel $k$ |
| Speed of light | $c$ | m/s ($2.998 \times 10^8$) | Exact constant |
| Geometric delay | $\tau_{p,s}$ | seconds | Per antenna $p$, source $s$ |
| Beam Jones (raw) | $J_p^{\alpha\gamma}$ | varies (depends on beam file) | Before normalization |
| Beam Jones (normalized) | $\tilde{J}_p^{\alpha\gamma}$ | dimensionless | After peak normalization: $\tilde{J} = J / J(\hat{\mathbf{z}})$ |
| Visibility | $V_{pq}^{\alpha\beta}$ | Jy | Thanks to peak beam normalization |
| Baseline vector | $\mathbf{b}_{pq}$ | meters | $\mathbf{b}_{pq} = \mathbf{r}_p - \mathbf{r}_q$ |
| Beam solid angle | $\Omega_B$ | sr (steradians) | $\Omega_B = \int |\tilde{J}|^2 d\Omega$ |

> **Index legend for the table above:** $p, q$ = antenna indices; $s$ = source index; $k$ = frequency channel index; $\alpha, \beta$ = feed indices ($X$, $Y$); $\gamma$ = sky polarization ($\hat{\theta}$, $\hat{\phi}$); $\hat{\mathbf{z}}$ = zenith direction.

### C.2 Conversion to Brightness Temperature

To convert visibilities from Jy to Kelvin (brightness temperature):

$$T_b = \frac{\lambda^2}{2 k_B \Omega_B} S$$

> **Index legend:**
>
> | Symbol | Meaning | Typical value / units |
> |--------|---------|----------------------|
> | $T_b$ | Brightness temperature | Kelvin (K) |
> | $\lambda$ | Wavelength: $\lambda = c / \nu$ | meters |
> | $k_B$ | Boltzmann constant | $1.381 \times 10^{-23}$ J/K |
> | $\Omega_B$ | Beam solid angle | steradians |
> | $S$ | Flux density | W m$^{-2}$ Hz$^{-1}$ (= $10^{-26}$ Jy) |

**This conversion is NOT performed by matvis** — matvis outputs are in Jy (or whatever units the input flux densities are in). The factor $\lambda^2 / (2 k_B \Omega_B)$ converts from flux density to temperature and depends on the beam shape.

### C.3 What matvis Does NOT Include

The following physical effects are **not** modeled by matvis. They must be added separately in a full simulation pipeline:

| Effect | Physical Description | Why Not Included |
|--------|---------------------|-----------------|
| **Ionospheric effects** | Faraday rotation, refraction, scintillation due to free electrons in Earth's ionosphere | Direction-dependent, time-variable; requires separate ionospheric model |
| **Tropospheric effects** | Delay, absorption, phase fluctuations from water vapor | Significant at high frequencies; separate atmospheric model needed |
| **Thermal noise** | Random voltage fluctuations from receiver electronics and sky temperature | Added post-simulation; depends on integration time, bandwidth, system temperature |
| **RFI** | Radio Frequency Interference from human-made sources | Highly variable; modeled separately or flagged |
| **Bandwidth smearing** | Decorrelation within a finite frequency channel width $\Delta\nu$ | Requires sub-channel sampling; matvis evaluates at channel centers |
| **Time smearing** | Decorrelation within a finite integration time $\Delta t$ | Requires sub-integration sampling; matvis evaluates at time centers |
| **Mutual coupling** | Electromagnetic interaction between nearby antennas | Requires full EM simulation; cannot be factored per-antenna |
| **Signal chain effects** | Cable delays, amplifier gains, bandpass shape, crosstalk | Added by a separate signal-chain simulator (e.g., `hera_sim.sigchain`) |
| **Extended sources** | Sources with angular structure (not point-like) | Would need pixelization or visibility-domain convolution |

All these effects are **multiplicative or additive corrections** that can be applied to the matvis output in a downstream pipeline.

---

## End of Documentation

This notebook has documented the complete mathematical and numerical framework of `matvis`. The key takeaways are:

1. **matvis implements the discrete RIME** as a sum over $N_{\text{src}}$ point sources (Section 1)
2. **Coordinate transforms** convert equatorial source positions to topocentric ENU using a $3\times3$ rotation matrix at each time step (Section 2)
3. **Beam evaluation** interpolates the Jones matrix $J_p^{\alpha\gamma}(\theta_s, \phi_s, \nu_k)$ at each source direction (Section 3)
4. **Beam normalization is PEAK** (at zenith), not area normalization: $\tilde{J} = J / J(\hat{\mathbf{z}})$ (Section 4)
5. **The coherency matrix** reduces to $\sqrt{I_s}$ for Stokes I sources (Section 5)
6. **Fringe computation** uses per-antenna delays $\tau_{p,s}$, with the baseline phase emerging from the outer product (Section 6)
7. **The core trick** is reformulating the RIME as a matrix product $\mathbf{V} = \mathbf{A}\mathbf{A}^H$ where $\mathbf{A} \in \mathbb{C}^{N_{\text{ant}} \times N_{\text{src}}}$ (Section 7)
8. **Full polarization** uses $2\times2$ Jones matrices and computes 4 visibility products: $XX, XY, YX, YY$ (Section 8)
9. **Time and frequency loops** are structured so that the geometric delay is computed once per time step and reused across frequencies (Section 9)
10. **GPU and CPU backends** compute the same equation; differences are only in floating-point precision and operation ordering (Section 10)
11. **The output is in the same units as the input flux densities** (typically Jy), thanks to peak beam normalization (Appendix C)

### Quick-Reference: The matvis Equation

$$\boxed{V_{pq}^{\alpha\beta}(\nu_k) = \left[\mathbf{A}^{\alpha}(\nu_k) \; \left(\mathbf{A}^{\beta}(\nu_k)\right)^H\right]_{pq}, \quad A_p^{\alpha}(s) = \tilde{J}_p^{\alpha}(\hat{\mathbf{s}}_s, \nu_k) \; \sqrt{I_s(\nu_k)} \; e^{-2\pi i \nu_k \hat{\mathbf{s}}_s \cdot \mathbf{r}_p / c}}$$

---
## Appendix D: Diagrams

The following diagrams illustrate the key concepts from the equations above. Each diagram is labeled with the mathematical symbols introduced in the text to build physical intuition.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(20, 13))

# ============================================================
# Diagram 1: Baseline Geometry & Fringe (top-left)
# ============================================================
ax = axes[0, 0]
ax.set_xlim(-1, 7)
ax.set_ylim(-1, 6)
ax.set_aspect('equal')
ax.set_title('Diagram 1: Baseline Geometry & Fringe', fontsize=12, fontweight='bold')

# Ground
ax.axhline(0, color='brown', lw=2, ls='--', alpha=0.5)
ax.text(3, -0.4, 'Ground', ha='center', fontsize=9, color='brown')

# Antenna p
ax.plot(1, 0, 'k^', markersize=15)
ax.annotate(r'Antenna $p$', (1, 0), (0.3, -0.7), fontsize=10, fontweight='bold')
ax.annotate(r'$\mathbf{r}_p$', (1, 0), (0.2, 0.3), fontsize=12, color='blue')

# Antenna q
ax.plot(5, 0, 'k^', markersize=15)
ax.annotate(r'Antenna $q$', (5, 0), (4.3, -0.7), fontsize=10, fontweight='bold')
ax.annotate(r'$\mathbf{r}_q$', (5, 0), (5.2, 0.3), fontsize=12, color='blue')

# Baseline vector
ax.annotate('', xy=(5, 0.15), xytext=(1, 0.15),
            arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax.text(3, 0.45, r'$\mathbf{b}_{pq} = \mathbf{r}_p - \mathbf{r}_q$', ha='center',
        fontsize=12, color='red', fontweight='bold')

# Source direction
src_x, src_y = 3, 5.5
ax.plot(src_x, src_y, '*', color='gold', markersize=20, markeredgecolor='orange', markeredgewidth=1)
ax.text(src_x + 0.3, src_y, r'Source $s$', fontsize=10)

# Direction vectors
ax.annotate('', xy=(src_x - 0.15, src_y - 0.3), xytext=(1, 0.3),
            arrowprops=dict(arrowstyle='->', color='green', lw=1.5, ls='--'))
ax.annotate('', xy=(src_x + 0.15, src_y - 0.3), xytext=(5, 0.3),
            arrowprops=dict(arrowstyle='->', color='green', lw=1.5, ls='--'))
ax.text(1.2, 3.0, r'$\hat{\mathbf{s}}_s$', fontsize=12, color='green', rotation=60)

# Delay annotation
ax.text(3, 2.2, r'$\tau_{p,s} = \frac{\hat{\mathbf{s}}_s \cdot \mathbf{r}_p}{c}$',
        fontsize=11, ha='center', color='purple',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lavender', alpha=0.8))

ax.axis('off')

# ============================================================
# Diagram 2: Jones Matrix Structure (top-center)
# ============================================================
ax = axes[0, 1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_title('Diagram 2: Jones Matrix Structure', fontsize=12, fontweight='bold')

# Draw the 2x2 matrix
box_x, box_y = 2, 3
box_w, box_h = 6, 4

# Grid lines
for i in range(3):
    ax.plot([box_x, box_x + box_w], [box_y + i * box_h / 2, box_y + i * box_h / 2], 'k-', lw=1.5)
for j in range(3):
    ax.plot([box_x + j * box_w / 2, box_x + j * box_w / 2], [box_y, box_y + box_h], 'k-', lw=1.5)

# Matrix elements
elements = [
    (r'$J_p^{X\hat{\theta}}$', 'Co-pol (dominant)'),
    (r'$J_p^{X\hat{\phi}}$', 'Cross-pol (small)'),
    (r'$J_p^{Y\hat{\theta}}$', 'Cross-pol (small)'),
    (r'$J_p^{Y\hat{\phi}}$', 'Co-pol (dominant)'),
]
positions = [(0.25, 0.75), (0.75, 0.75), (0.25, 0.25), (0.75, 0.25)]
colors = ['#2196F3', '#FF9800', '#FF9800', '#2196F3']

for (elem, desc), (fx, fy), color in zip(elements, positions, colors):
    cx = box_x + fx * box_w
    cy = box_y + fy * box_h
    ax.text(cx, cy + 0.25, elem, ha='center', va='center', fontsize=13, color=color, fontweight='bold')
    ax.text(cx, cy - 0.35, desc, ha='center', va='center', fontsize=7, color='gray')

# Row labels
ax.text(box_x - 0.5, box_y + 0.75 * box_h, r'$X$ feed $(\alpha=X)$', ha='right', va='center', fontsize=10, color='blue')
ax.text(box_x - 0.5, box_y + 0.25 * box_h, r'$Y$ feed $(\alpha=Y)$', ha='right', va='center', fontsize=10, color='blue')

# Column labels
ax.text(box_x + 0.25 * box_w, box_y + box_h + 0.4, r'$\hat{\theta}$ sky $(\gamma)$', ha='center', fontsize=10, color='green')
ax.text(box_x + 0.75 * box_w, box_y + box_h + 0.4, r'$\hat{\phi}$ sky $(\gamma)$', ha='center', fontsize=10, color='green')

# Title for matrix
ax.text(5, 8.5, r'Jones Matrix $\mathbf{J}_p$', ha='center', fontsize=13, fontweight='bold')
ax.text(5, 7.8, r'Rows = antenna feeds $(\alpha)$', ha='center', fontsize=9, color='blue')
ax.text(5, 7.3, r'Columns = sky polarizations $(\gamma)$', ha='center', fontsize=9, color='green')

# Physical explanation
# ax.text(5, 1.5, r'$\begin{pmatrix} v_p^X \\ v_p^Y \end{pmatrix} = \mathbf{J}_p \begin{pmatrix} E^{\hat{\theta}} \\ E^{\hat{\phi}} \end{pmatrix}$',
#         ha='center', fontsize=12,
#         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

ax.axis('off')

# ============================================================
# Diagram 3: Per-Antenna Vector Construction (top-right)
# ============================================================
ax = axes[0, 2]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_title('Diagram 3: Per-Antenna Vector $A_p(s)$', fontsize=12, fontweight='bold')

# Flow diagram: beam × sqrt(I) × fringe = A
y_start = 8
box_params = dict(boxstyle='round,pad=0.4', alpha=0.9, lw=1.5)

components = [
    (r'$\tilde{J}_p^\alpha(\hat{\mathbf{s}}_s, \nu_k)$', 'Beam\n(normalized)', '#E3F2FD', '#1565C0'),
    (r'$\sqrt{I_s(\nu_k)}$', 'Source\nflux', '#FFF3E0', '#E65100'),
    (r'$e^{-2\pi i \nu_k \tau_{p,s}}$', 'Fringe\nphasor', '#E8F5E9', '#2E7D32'),
]

for idx, (formula, label, facecolor, edgecolor) in enumerate(components):
    y = y_start - idx * 2.5
    ax.text(5, y, formula, ha='center', va='center', fontsize=13,
            bbox=dict(boxstyle='round,pad=0.5', facecolor=facecolor, edgecolor=edgecolor, lw=2))
    ax.text(8.5, y, label, ha='center', va='center', fontsize=9, color='gray')
    if idx < 2:
        ax.annotate('', xy=(5, y - 0.7), xytext=(5, y - 0.5),
                    arrowprops=dict(arrowstyle='-', color='black', lw=1))
        ax.text(6.2, y - 0.9, r'$\times$', fontsize=14, ha='center', va='center')

# Result
ax.text(5, 1.2, r'$A_p^\alpha(s) = \tilde{J}_p^\alpha \cdot \sqrt{I_s} \cdot e^{-2\pi i \nu_k \tau_{p,s}}$',
        ha='center', va='center', fontsize=12, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#FFEBEE', edgecolor='#C62828', lw=2))
ax.annotate('', xy=(5, 1.9), xytext=(5, 2.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

ax.axis('off')

# ============================================================
# Diagram 4: Matrix Multiply V = A A^H (bottom-left)
# ============================================================
ax = axes[1, 0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.set_title(r'Diagram 4: Outer Product $\mathbf{V} = \mathbf{A} \mathbf{A}^H$', fontsize=12, fontweight='bold')

# Matrix A
rect_a = mpatches.FancyBboxPatch((0.3, 2.5), 1.5, 3, boxstyle="round,pad=0.1",
                                   facecolor='#E3F2FD', edgecolor='#1565C0', lw=2)
ax.add_patch(rect_a)
ax.text(1.05, 4.0, r'$\mathbf{A}$', ha='center', va='center', fontsize=16, fontweight='bold', color='#1565C0')
ax.text(1.05, 3.0, r'$N_{\rm ant}$' + '\n' + r'$\times N_{\rm src}$', ha='center', va='center', fontsize=8)

# × symbol
ax.text(2.3, 4.0, r'$\times$', ha='center', va='center', fontsize=18)

# Matrix A^H
rect_ah = mpatches.FancyBboxPatch((2.8, 2.5), 3, 1.5, boxstyle="round,pad=0.1",
                                    facecolor='#FFF3E0', edgecolor='#E65100', lw=2)
ax.add_patch(rect_ah)
ax.text(4.3, 3.25, r'$\mathbf{A}^H$', ha='center', va='center', fontsize=16, fontweight='bold', color='#E65100')
ax.text(4.3, 2.8, r'$N_{\rm src} \times N_{\rm ant}$', ha='center', va='center', fontsize=8)

# = symbol
ax.text(6.3, 4.0, r'$=$', ha='center', va='center', fontsize=18)

# Matrix V
rect_v = mpatches.FancyBboxPatch((6.8, 2.5), 3, 3, boxstyle="round,pad=0.1",
                                   facecolor='#E8F5E9', edgecolor='#2E7D32', lw=2)
ax.add_patch(rect_v)
ax.text(8.3, 4.0, r'$\mathbf{V}$', ha='center', va='center', fontsize=16, fontweight='bold', color='#2E7D32')
ax.text(8.3, 3.0, r'$N_{\rm ant}$' + '\n' + r'$\times N_{\rm ant}$', ha='center', va='center', fontsize=8)

# Annotations
ax.text(5, 1.2, r'$V_{pq} = \sum_s A_p(s) \cdot A_q^*(s)$', ha='center', fontsize=12,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
ax.text(5, 0.3, r'Element $(p,q)$ = dot product of row $p$ with conjugate of row $q$',
        ha='center', fontsize=9, color='gray')

# Labels
ax.text(1.05, 6.0, 'rows = antennas\ncols = sources', ha='center', fontsize=8, color='#1565C0')
ax.text(8.3, 6.0, 'rows = ant $p$\ncols = ant $q$', ha='center', fontsize=8, color='#2E7D32')
ax.text(5, 7.2, r'Hermitian: $V_{pq} = V_{qp}^*$', ha='center', fontsize=10, color='purple')

ax.axis('off')

# ============================================================
# Diagram 5: Coordinate Frames (bottom-center)
# ============================================================
ax = axes[1, 1]
ax.set_xlim(-2, 4)
ax.set_ylim(-1, 5)
ax.set_aspect('equal')
ax.set_title('Diagram 5: Coordinate Frames', fontsize=12, fontweight='bold')

# Origin
ax.plot(0, 0, 'ko', markersize=5)

# ENU axes
ax.annotate('', xy=(3, 0), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax.text(3.2, 0, r'East ($l$)', fontsize=10, color='red', va='center')

ax.annotate('', xy=(0, 3), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='blue', lw=2))
ax.text(0.2, 3.2, r'North ($m$)', fontsize=10, color='blue')

ax.annotate('', xy=(1.5, 1.5), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='green', lw=2))
ax.text(1.7, 1.7, r'Up ($n$)', fontsize=10, color='green')

# Zenith
ax.plot(0, 4.3, '*', color='gold', markersize=15)
ax.text(0.3, 4.3, r'Zenith $\hat{\mathbf{z}}$', fontsize=10, va='center')
ax.text(0.3, 3.8, r'$\theta = 0^\circ$', fontsize=8, color='gray')

# Source at angle
theta_src = np.radians(35)
r_src = 3.5
sx, sy = r_src * np.sin(theta_src), r_src * np.cos(theta_src)
ax.plot(sx, sy, '*', color='orange', markersize=12)
ax.text(sx + 0.3, sy, r'$\hat{\mathbf{s}}_s$', fontsize=11, color='orange')

# Zenith angle arc
theta_arr = np.linspace(0, theta_src, 30)
r_arc = 2.0
ax.plot(r_arc * np.sin(theta_arr), r_arc * np.cos(theta_arr), 'k--', lw=1)
ax.text(0.7, 2.2, r'$\theta_s$', fontsize=11, color='black')

# Antenna position
ax.plot(2.0, 0, 'k^', markersize=12)
ax.annotate(r'$\mathbf{r}_p$', (2.0, 0), (2.0, -0.5), fontsize=11, color='purple', ha='center')

# Frame labels
ax.text(1.0, -0.8, 'Topocentric ENU Frame', ha='center', fontsize=10, fontweight='bold')

ax.axis('off')

# ============================================================
# Diagram 6: Beam Pattern with Normalization (bottom-right)
# ============================================================
ax = axes[1, 2]

theta = np.linspace(-np.pi/2, np.pi/2, 500)
# Gaussian beam
sigma = np.radians(20)
beam = np.exp(-theta**2 / (2 * sigma**2))

# Normalized beam
beam_norm = beam / beam.max()

ax.fill_between(np.degrees(theta), beam_norm, alpha=0.2, color='blue')
ax.plot(np.degrees(theta), beam_norm, 'b-', lw=2, label=r'$\tilde{J}(\theta) = J(\theta)/J(0)$')

# Mark zenith
ax.axvline(0, color='red', ls=':', lw=1.5, alpha=0.7)
ax.plot(0, 1.0, 'ro', markersize=10, zorder=5)
ax.annotate(r'Zenith $\hat{\mathbf{z}}$: $\tilde{J}=1$', (0, 1.0), (15, 0.95),
            fontsize=10, color='red', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

# Mark a source
theta_src = 30
beam_src = np.exp(-(np.radians(theta_src))**2 / (2 * sigma**2))
ax.plot(theta_src, beam_src, 'g*', markersize=15, zorder=5)
ax.annotate(r'Source $s$: $\tilde{J}(\theta_s)$', (theta_src, beam_src),
            (theta_src + 10, beam_src + 0.15), fontsize=10, color='green',
            arrowprops=dict(arrowstyle='->', color='green', lw=1.5))

# Horizon
ax.axvline(-90, color='brown', ls='--', lw=1.5, alpha=0.7)
ax.axvline(90, color='brown', ls='--', lw=1.5, alpha=0.7)
ax.text(-88, 0.5, 'Horizon', rotation=90, fontsize=9, color='brown', va='center')
ax.text(80, 0.5, 'Horizon', rotation=90, fontsize=9, color='brown', va='center')

# Labels
ax.set_xlabel(r'Zenith Angle $\theta$ (degrees)', fontsize=11)
ax.set_ylabel(r'Normalized Beam $\tilde{J}(\theta)$', fontsize=11)
ax.set_title('Diagram 6: Beam Pattern (Peak Normalized)', fontsize=12, fontweight='bold')
ax.set_xlim(-95, 95)
ax.set_ylim(-0.05, 1.15)
ax.legend(fontsize=10, loc='upper left')

# Annotation about normalization
ax.text(0, -0.15, r'Peak norm: $\tilde{J}(\hat{\mathbf{z}}) \equiv 1$ $\Rightarrow$ 1 Jy source at zenith gives 1 Jy visibility',
        ha='center', fontsize=8, color='purple', style='italic',
        bbox=dict(boxstyle='round', facecolor='lavender', alpha=0.7))

plt.tight_layout()
# plt.savefig('/lustre/aoc/projects/hera/rchandra/H6C_Validation_Stats/validation-sim/notebooks/matvis_diagrams.png', dpi=150, bbox_inches='tight')
plt.show()
print("Diagrams saved.")

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ============================================================
# Diagram 7: Visibility Matrix (Polarization Products)
# ============================================================
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_title(r'Diagram 7: Visibility Matrix $\mathbf{V}_{pq}$', fontsize=13, fontweight='bold')

# 2x2 grid
bx, by, bw, bh = 2, 2, 6, 6
for i in range(3):
    ax.plot([bx, bx + bw], [by + i * bh / 2, by + i * bh / 2], 'k-', lw=2)
    ax.plot([bx + i * bw / 2, bx + i * bw / 2], [by, by + bh], 'k-', lw=2)

# Elements with colors
entries = [
    (0.25, 0.75, r'$V_{pq}^{XX}$', r'$\langle v_p^X v_q^{X*}\rangle$', r'$\sim I+Q$', '#2196F3'),
    (0.75, 0.75, r'$V_{pq}^{XY}$', r'$\langle v_p^X v_q^{Y*}\rangle$', r'$\sim U+iV$', '#FF9800'),
    (0.25, 0.25, r'$V_{pq}^{YX}$', r'$\langle v_p^Y v_q^{X*}\rangle$', r'$\sim U-iV$', '#FF9800'),
    (0.75, 0.25, r'$V_{pq}^{YY}$', r'$\langle v_p^Y v_q^{Y*}\rangle$', r'$\sim I-Q$', '#4CAF50'),
]
for fx, fy, label, corr, stokes, color in entries:
    cx, cy = bx + fx * bw, by + fy * bh
    ax.text(cx, cy + 0.6, label, ha='center', va='center', fontsize=14, fontweight='bold', color=color)
    ax.text(cx, cy, corr, ha='center', va='center', fontsize=10, color='gray')
    ax.text(cx, cy - 0.6, stokes, ha='center', va='center', fontsize=10, color='purple')

# Row/col labels
ax.text(bx - 0.3, by + 0.75 * bh, r'$\alpha = X$', ha='right', va='center', fontsize=11, color='blue')
ax.text(bx - 0.3, by + 0.25 * bh, r'$\alpha = Y$', ha='right', va='center', fontsize=11, color='blue')
ax.text(bx + 0.25 * bw, by + bh + 0.4, r'$\beta = X$', ha='center', fontsize=11, color='green')
ax.text(bx + 0.75 * bw, by + bh + 0.4, r'$\beta = Y$', ha='center', fontsize=11, color='green')

ax.text(bx - 0.8, by + bh / 2, r'Feed on $p$ $\rightarrow$', ha='center', va='center',
        fontsize=9, color='blue', rotation=90)
ax.text(bx + bw / 2, by + bh + 1.0, r'Feed on $q$ $\rightarrow$', ha='center',
        fontsize=9, color='green')

ax.text(5, 0.5, r'4 correlations $\rightarrow$ 4 Stokes parameters: $I, Q, U, V$',
        ha='center', fontsize=10, style='italic',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

ax.axis('off')

# ============================================================
# Diagram 8: matvis Pipeline Overview
# ============================================================
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_title('Diagram 8: matvis Computation Pipeline', fontsize=13, fontweight='bold')

steps = [
    (r'Sky Catalog: $(\hat{\mathbf{s}}_s, I_s)$', '#E8EAF6', '#283593'),
    (r'Coordinate Transform: $\hat{\mathbf{s}}_{\rm top} = R(t) \hat{\mathbf{s}}_{\rm eq}$', '#E3F2FD', '#1565C0'),
    (r'Horizon Cut: $n_s > 0$', '#E0F7FA', '#00695C'),
    (r'Beam Eval: $J_p(\theta_s, \phi_s, \nu_k)$', '#E8F5E9', '#2E7D32'),
    (r'Normalize: $\tilde{J} = J / J(\hat{\mathbf{z}})$', '#FFF8E1', '#F57F17'),
    (r'Delay: $\tau_{p,s} = \hat{\mathbf{s}}_s \cdot \mathbf{r}_p / c$', '#FFF3E0', '#E65100'),
    (r'Build $A_p(s) = \tilde{J}_p \sqrt{I_s} \, e^{-2\pi i \nu_k \tau_{p,s}}$', '#FCE4EC', '#AD1457'),
    (r'Outer Product: $\mathbf{V} = \mathbf{A}\mathbf{A}^H$', '#FFEBEE', '#B71C1C'),
]

y_positions = np.linspace(9, 1, len(steps))
for idx, ((text, facecolor, edgecolor), y) in enumerate(zip(steps, y_positions)):
    ax.text(5, y, text, ha='center', va='center', fontsize=9.5,
            bbox=dict(boxstyle='round,pad=0.4', facecolor=facecolor, edgecolor=edgecolor, lw=1.5))
    if idx < len(steps) - 1:
        ax.annotate('', xy=(5, y_positions[idx + 1] + 0.35), xytext=(5, y - 0.35),
                    arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

# Loop annotations
ax.annotate('', xy=(9.5, y_positions[0] + 0.3), xytext=(9.5, y_positions[-1] - 0.3),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1, ls='--'))
ax.text(9.7, 5, r'Loop over $t, k$', fontsize=8, color='gray', rotation=90, ha='center', va='center')

ax.axis('off')

plt.tight_layout()
plt.savefig('/lustre/aoc/projects/hera/rchandra/H6C_Validation_Stats/validation-sim/notebooks/matvis_diagrams_2.png', dpi=150, bbox_inches='tight')
plt.show()
print("Diagrams 7-8 saved.")

---
## Appendix E: Why matvis Cannot Handle Negative Sky Brightness — and How to Fix It

*(Added 2026-08-03. Code references are to the installed package `matvis 1.3.2` at*
`.../envs/myenv_validation_1/lib/python3.10/site-packages/matvis/`*)*

### E.1 The factorization at the heart of matvis

The (unpolarized) RIME that matvis evaluates for each time and frequency is

$$
V_{pq} \;=\; \sum_{s} \frac{I_s}{2}\, A_p(\hat{\mathbf{s}}_s)\, A_q^*(\hat{\mathbf{s}}_s)\,
e^{\,i\,2\pi\nu\,(\mathbf{b}_q-\mathbf{b}_p)\cdot\hat{\mathbf{s}}_s/c},
$$

a sum over sources $s$ with brightness $I_s$, per-antenna beam response $A_p$, and geometric
phase $\tau_p = 2\pi\nu\,\mathbf{b}_p\cdot\hat{\mathbf{s}}_s/c$.

The entire speed advantage of matvis comes from noticing that this double-indexed sum is a
**Gram matrix**: it *factorizes* into a product of per-antenna quantities. matvis splits the
flux **symmetrically** between the two antennas of each baseline by taking its square root,

$$
z_{p,s} \;\equiv\; A_p(\hat{\mathbf{s}}_s)\,\sqrt{\tfrac{I_s}{2}}\;e^{\,i\tau_{p,s}},
\qquad\Longrightarrow\qquad
V_{pq} \;=\; \sum_s z_{p,s}^{*}\, z_{q,s}
\;=\; \bigl(Z^{*} Z^{T}\bigr)_{pq},
$$

so that **all** baselines for one time/frequency are obtained from a *single* matrix–matrix
multiplication (one BLAS GEMM call), at cost $\mathcal{O}(N_{\rm ant}^2 N_{\rm src})$ with a
tiny constant. This is exactly what the code does, in three steps:

1. **The square root is taken up front** — `cpu/cpu.py`, line 166 (identically `gpu/gpu.py`, line 115):
   ```python
   coords = coord_method(
       flux=np.sqrt(0.5 * I_sky),     # <-- sqrt of a REAL (float32/64) array
       ...
   )
   ```
2. **The Z matrix is assembled** — `core/getz.py`, line 80 (`ZMatrixCalc.__call__`):
   ```python
   exptau *= sqrt_flux                 # Z = A * sqrt(I/2) * exp(i*tau)
   ```
3. **The Gram product forms all visibilities at once** — `cpu/matprod.py`, line 21 (`CPUMatMul.compute`):
   ```python
   v = z.conj().dot(z.T)               # V = Z* Z^T   (a Gram / outer-product matrix)
   ```

### E.2 Where negative brightness breaks it, numerically

`I_sky` enters *only* through `np.sqrt(0.5 * I_sky)` acting on a **real-typed** array.
For any source with $I_s < 0$:

$$
\sqrt{I_s} \notin \mathbb{R}
\quad\Longrightarrow\quad
\texttt{np.sqrt}(I_s) = \texttt{NaN} \ \ (\text{with a RuntimeWarning}).
$$

That NaN multiplies into the corresponding column of $Z$ (step 2), and because step 3 is a
full matrix product summing over *all* sources, a **single** negative-brightness source
poisons **every** visibility for **every** baseline, feed, and time: the output is all-NaN.
There is no explicit validation of the sign of `I_sky` anywhere in the package — the failure
is silent apart from the NumPy invalid-value warning.

(The same square-root pattern appears in the unpolarized beam path, `cpu/beams.py`, line 46:
`interp_beam = np.sqrt(interp_beam[0, 0, 0, :])` — a *power* beam that interpolates to small
negative values in sidelobe nulls triggers the identical NaN mechanism.)

### E.3 Why this is structural, not just a missing `abs()`

One might hope to fix it by allowing a complex square root, $\sqrt{I_s} \to i\sqrt{|I_s|}$
for $I_s<0$. **This cannot work inside the symmetric factorization.** The Gram product
conjugates one factor:

$$
V_{pq} \;=\; \sum_s z_{p,s}^{*} z_{q,s}
\;\propto\; \sum_s \bigl(\sqrt{I_s}\bigr)^{*}\bigl(\sqrt{I_s}\bigr)\,(\cdots)
\;=\; \sum_s \bigl|\sqrt{I_s}\bigr|^2\,(\cdots)
\;=\; \sum_s |I_s|\,(\cdots),
$$

because $(i)^{*}(i) = (-i)(i) = +1$. The conjugation **destroys the sign**: a negative
source would silently contribute $+|I_s|$ — a wrong answer rather than a NaN, which is worse.

The deeper statement: for any matrix $Z$, the Gram matrix $V = Z^* Z^T$ is Hermitian
**positive semi-definite** by construction,

$$
\mathbf{w}^{\dagger} V^{T} \mathbf{w} = \lVert Z^{\dagger}\mathbf{w}\rVert^2 \;\ge\; 0
\quad \forall\, \mathbf{w},
$$

whereas the true visibility matrix of a sky containing negative brightness,
$V = \sum_s I_s\, \mathbf{a}_s \mathbf{a}_s^{\dagger}$ (with
$\mathbf{a}_s$ the array response vector to source $s$), is Hermitian but generally
**indefinite**. An indefinite matrix *cannot be written as $Z^*Z^T$ for any $Z$
whatsoever* — so no choice of square root, real or complex, rescues the single-Gram-product
formulation. Physically: $ZZ^{\dagger}$ is the covariance (coherency) matrix of an
incoherent electric field, and an intensity is a variance, which is non-negative.
Negative brightness only arises in *derived* skies — mean/monopole-subtracted diffuse maps,
residual (data − model) skies, CLEAN component differences — which are precisely the cases
that occur in validation work.

### E.4 The fixes

**Fix 1 — Split the sky by sign and use linearity (no code changes; recommended).**
The RIME is linear in $I_s$. Decompose

$$
I_s = I_s^{+} - I_s^{-},\qquad
I_s^{+} = \max(I_s,0)\ \ge 0,\qquad
I_s^{-} = \max(-I_s,0)\ \ge 0,
$$

run matvis once on each non-negative sky, and subtract:

$$
V = V[I^{+}] \;-\; V[I^{-}].
$$

This is *exact* (in exact arithmetic). Since each run only needs the sources with non-zero
flux of its sign, the **total source count across the two runs equals the original**, so the
dominant $\mathcal{O}(N_{\rm ant}^2 N_{\rm src})$ cost is essentially unchanged — you pay
only a second setup overhead. One caveat for `precision=1` (float32): if $V[I^+]$ and
$V[I^-]$ are large and nearly cancel, the difference loses relative precision
(catastrophic cancellation); use `precision=2` for residual-type skies.

**Fix 2 — Asymmetric (signed) factorization (small patch to matvis).**
Nothing forces the two factors to be identical. Put the sign in *one* factor only:

$$
\tilde{z}_{p,s} = \operatorname{sign}(I_s)\sqrt{\tfrac{|I_s|}{2}}\,A_p e^{i\tau_p},
\qquad
z_{q,s} = \sqrt{\tfrac{|I_s|}{2}}\,A_q e^{i\tau_q},
\qquad
V_{pq} = \sum_s \tilde{z}_{p,s}^{*}\, z_{q,s},
$$

so the source weight becomes $\operatorname{sign}(I_s)\,|I_s|/2 = I_s/2$, exactly as
required. Equivalently, $V = Z^{*}\,\mathrm{diag}(\operatorname{sign} I)\,Z^{T}$. Concretely:

```python
# cpu/cpu.py  (line 166)
flux = np.sqrt(0.5 * np.abs(I_sky))        # magnitudes only -> always real
sgn  = np.sign(I_sky)                      # carry the sign separately

# cpu/matprod.py (line 21) — one GEMM of the SAME size, so same O() cost:
v = (z * sgn).conj().dot(z.T)              # V = Z* diag(sgn) Z^T
```

The GEMM dimensions are unchanged, so throughput is the same; the costs are one extra
elementwise multiply and (if not done in place) one extra copy of $Z$. Splitting the
*magnitude* symmetrically ($\sqrt{|I|}$ on both sides) rather than putting $|I_s|$ wholly on
one side preserves float32 dynamic range. The result is analytically Hermitian but no longer
PSD-by-construction — which is exactly what an indefinite sky requires. (The `CPUVectorLoop`
/ GPU `matprod` variants and the beam square root in `cpu/beams.py` need the analogous
one-line changes.)

**Summary.** The negative-brightness limitation is a direct consequence of matvis's
defining optimization — writing the RIME as a self-adjoint Gram product $Z^{*}Z^{T}$ with the
flux split as $\sqrt{I_s}$ between the conjugated factors. Real square roots of negative
numbers give NaN; complex ones lose the sign under conjugation; and no factorization of this
symmetric form can represent an indefinite visibility matrix. For the H6C validation
workflow the clean solution is **Fix 1**: simulate the positive and negative parts of the
sky separately with the unmodified package and subtract the resulting visibilities
(at `precision=2` for strongly cancelling residual skies).


In [ ]:
# Appendix E demo: reproduce the failure mechanism and verify both fixes,
# using the exact factorization matvis uses (numpy only, tiny toy array).
import numpy as np

rng = np.random.default_rng(42)
nant, nsrc = 3, 5
freq = 150e6                                  # Hz
c = 299792458.0

antpos = rng.uniform(-50, 50, (nant, 3))      # meters
crd = rng.normal(size=(3, nsrc))              # unit source vectors
crd /= np.linalg.norm(crd, axis=0)
A = rng.uniform(0.2, 1.0, (nant, nsrc))       # toy (real) beam values
I_sky = np.array([1.0, 2.5, -1.5, 0.7, -0.3]) # sky WITH negative brightness

tau = 2j * np.pi * freq / c * (antpos @ crd)  # phase per antenna/source


def matvis_gram(I):
    """V = Z* Z^T with Z = A sqrt(I/2) exp(tau) -- what matvis actually does."""
    with np.errstate(invalid="ignore"):
        z = A * np.sqrt(0.5 * I) * np.exp(tau)
    return z.conj() @ z.T


def rime_direct(I):
    """Direct RIME sum -- the ground truth."""
    V = np.zeros((nant, nant), complex)
    for p in range(nant):
        for q in range(nant):
            V[p, q] = np.sum(0.5 * I * A[p] * A[q] * np.exp(tau[q] - tau[p]))
    return V


V_true = rime_direct(I_sky)

# --- 1. matvis symmetric factorization: NaNs everywhere -----------------------
V_nan = matvis_gram(I_sky)
print("matvis factorization, real sqrt of negative flux -> all NaN:")
print(V_nan, "\n")

# --- 2. complex sqrt does NOT fix it: conjugation destroys the sign ----------
z_cplx = A * np.sqrt(0.5 * I_sky.astype(complex)) * np.exp(tau)
V_abs = z_cplx.conj() @ z_cplx.T
print("complex sqrt -> silently simulates |I| instead of I "
      "(matches RIME of abs(I), not of I):")
print("  max |V_cplx - V_true(|I|)| =",
      np.abs(V_abs - rime_direct(np.abs(I_sky))).max())
print("  max |V_cplx - V_true( I )| =",
      np.abs(V_abs - V_true).max(), " <- WRONG answer\n")

# --- 3. Fix 1: split sky by sign, run twice, subtract ------------------------
V_split = matvis_gram(np.clip(I_sky, 0, None)) - matvis_gram(np.clip(-I_sky, 0, None))
print("Fix 1 (I+ / I- split):      max error =", np.abs(V_split - V_true).max())

# --- 4. Fix 2: signed (asymmetric) factorization, single GEMM ----------------
z_mag = A * np.sqrt(0.5 * np.abs(I_sky)) * np.exp(tau)
V_signed = (z_mag * np.sign(I_sky)).conj() @ z_mag.T
print("Fix 2 (signed Z* diag(s) Z^T): max error =", np.abs(V_signed - V_true).max())

# --- 5. the structural reason: true V is indefinite, a Gram matrix can't be --
eigs = np.linalg.eigvalsh(V_true)
print("\neigenvalues of true V (note the negative one -> V is indefinite,"
      "\nhence NOT expressible as Z* Z^T for ANY Z):")
print(eigs)


---
## Appendix F: Fix 2 Implementation Plan — Exact Edits for the Signed (Asymmetric) Factorization

*(Added 2026-08-03. This is the complete, file-by-file edit list for implementing Fix 2 of
Appendix E. All line numbers refer to the installed* `matvis 1.3.2` *at*
`.../envs/myenv_validation_1/lib/python3.10/site-packages/matvis/`*. Apply the edits to an
**editable clone** — e.g. `pip download matvis==1.3.2 --no-deps --no-binary :all:` or clone the
matvis GitHub repo at tag `v1.3.2`, then `pip install -e .` in a cloned conda env — never to
`site-packages` in place, or a pip/conda update will silently revert the fix.)*

### F.1 Design recap

Fix 2 carries the sign of the flux in **one** factor of the matrix product only:

$$
V \;=\; Z^{*}\,\mathrm{diag}\!\bigl(\operatorname{sign} I\bigr)\,Z^{T},
\qquad
Z \text{ built from the signed root } s_s = \operatorname{sign}(I_s)\sqrt{|I_s|/2},
$$

so each source contributes $\operatorname{sign}(I_s)\cdot|I_s|/2 = I_s/2$ exactly. Three
properties make the patch small and safe:

1. **The signed root rides through `core/coords.py` untouched** — that module only stores
   (line 65), slices (112), horizon-masks (141), and zero-pads (144) the flux; it never does
   arithmetic on it. Chunking and masking therefore need **zero changes**.
2. **A `negative_sky` guard** (computed once from `I_sky`) keeps every all-non-negative sky on
   the exact legacy code path — outputs are **bit-for-bit identical** to unpatched v1.3.2.
3. **The GPU already has the needed primitive**: `gpu/_cublas.py` implements
   `complex_matmul(a, b) = a.conj() @ b.T` via general `cgemm`/`zgemm` (not a Hermitian
   rank-k `cherk`, which would have forced `a == b`), and `zdotz(a)` is just
   `complex_matmul(a, a)`.

### F.2 Summary table of all edits

| # | File | Line(s) | Kind | Purpose |
|---|------|---------|------|---------|
| 1 | `cpu/cpu.py` | 164 (insert) | code | compute `negative_sky` flag once |
| 2 | `cpu/cpu.py` | 166 | code | signed sqrt of flux |
| 3 | `cpu/cpu.py` | 229–243 | code | per-chunk sign vector; pass to matprod |
| 4 | `core/matprod.py` | 68–80 | signature | abstract `compute()` gains `sgn=None` |
| 5 | `core/matprod.py` | 82–100 | signature | `__call__()` gains and forwards `sgn=None` |
| 6 | `cpu/matprod.py` | 11–21 | code | `CPUMatMul`: signed GEMM |
| 7 | `cpu/matprod.py` | 38–51 | code | `CPUVectorDot`: signed per-pair dot |
| 8 | `gpu/gpu.py` | 113 (insert) | code | `negative_sky` flag (GPU mirror of #1) |
| 9 | `gpu/gpu.py` | 115 | code | signed sqrt (GPU mirror of #2) |
| 10 | `gpu/gpu.py` | 193–229 | code | per-chunk sign vector on GPU; pass to matprod |
| 11 | `gpu/matprod.py` | 32–36 | code | `GPUMatMul`: two-matrix GEMM when signed |
| 12 | `gpu/matprod.py` | 81–89 | code | `GPUVectorDot`: signed per-pair GEMM |
| 13 | `cpu/cpu.py` | 62–68 | docstring | `I_sky` may be negative |
| 14 | `wrapper.py` | `fluxes` param docs | docstring | same note in public API |

### F.3 Exact before/after per edit

#### `cpu/cpu.py`

**Edits 1–2 — flag + signed sqrt (lines 162–166):**
```python
# BEFORE
    coord_method = CoordinateRotation._methods[coord_method]

    coord_method_params = coord_method_params or {}
    coords = coord_method(
        flux=np.sqrt(0.5 * I_sky),

# AFTER
    coord_method = CoordinateRotation._methods[coord_method]

    coord_method_params = coord_method_params or {}
    negative_sky = bool(np.any(I_sky < 0))
    coords = coord_method(
        flux=np.sign(I_sky) * np.sqrt(0.5 * np.abs(I_sky)),
```

**Edit 3 — time/chunk loop (lines 228–243):**
```python
# BEFORE
        for c in range(nchunks):
            crd_top, flux_sqrt, nn = coords.select_chunk(c, t)
            ...
            z = zcalc(flux_sqrt, A, exptau, bmfunc.beam_idx)
            logdebug("Z", z[..., :nn])

            matprod(z, c)

# AFTER
        for c in range(nchunks):
            crd_top, flux_sqrt, nn = coords.select_chunk(c, t)
            sgn = np.tile(np.sign(flux_sqrt), nax) if negative_sky else None
            ...
            z = zcalc(flux_sqrt, A, exptau, bmfunc.beam_idx)
            logdebug("Z", z[..., :nn])

            matprod(z, c, sgn=sgn)
```
Notes: `nax` is in scope (line 145). `flux_sqrt` is the padded `nsrc_alloc`-length chunk
array, so `np.tile(sgn, nax)` has length `nax*nsrc_alloc == z.shape[1]`. `getz.py` flattens
the last axis of `Z` as `(nax, nsrc)` — index `ax*nsrc + s` — so `np.tile` (whole-array
repetition) is the correct expansion, **not** `np.repeat`. Padding zeros give `sign=0`, but
those `Z` columns are already zero, so they contribute nothing either way.

**Edit 13 — docstring (lines 62–68), append to the `I_sky` parameter description:**
```
        Values may be negative (e.g. residual or mean-subtracted sky models);
        the sign is carried exactly through the visibility computation.
```

#### `core/matprod.py`

**Edit 4 — abstract `compute` (lines 68–80):**
```python
# BEFORE
    @abstractmethod
    def compute(self, z: np.ndarray, out: np.ndarray):

# AFTER
    @abstractmethod
    def compute(self, z: np.ndarray, out: np.ndarray, sgn: np.ndarray | None = None):
```
(add to docstring: *sgn — optional per-column source-sign vector, length `z.shape[-1]`;
`None` means all-positive sky and must reproduce the legacy path exactly*).

**Edit 5 — `__call__` (lines 82–100):**
```python
# BEFORE
    def __call__(self, z: np.ndarray, chunk: int) -> np.ndarray:
        ...
        self.compute(z, out=self.vis[chunk])

# AFTER
    def __call__(self, z: np.ndarray, chunk: int, sgn: np.ndarray | None = None) -> np.ndarray:
        ...
        self.compute(z, out=self.vis[chunk], sgn=sgn)
```
Existing positional callers (`matprod(z, c)`) remain valid — default `sgn=None`.

#### `cpu/matprod.py`

**Edit 6 — `CPUMatMul.compute` (lines 11–21):**
```python
# BEFORE
    def compute(self, z: np.ndarray, out: np.ndarray) -> np.ndarray:
        ...
        v = z.conj().dot(z.T)

# AFTER
    def compute(self, z, out, sgn=None):
        ...
        zs = z if sgn is None else z * sgn        # unsigned copy: sgn * signed-sqrt = |sqrt|
        v = z.conj().dot(zs.T)                    # V = Z* diag(sgn) Z^T ; per-source weight = I/2
```

**Edit 7 — `CPUVectorDot.compute` (lines 38–51):**
```python
# BEFORE
    def compute(self, z: np.ndarray, out: np.ndarray) -> np.ndarray:
        z = z.reshape((self.nant, self.nfeed, -1))

        for i, (ai, aj) in enumerate(self.antpairs):
            out[i] = z[aj].dot(z[ai].conj().T)  # dot(z[aj].T)

# AFTER
    def compute(self, z, out, sgn=None):
        z = z.reshape((self.nant, self.nfeed, -1))
        zs = z if sgn is None else z * sgn

        for i, (ai, aj) in enumerate(self.antpairs):
            out[i] = zs[aj].dot(z[ai].conj().T)  # dot(z[aj].T)
```
(`sgn`, length `nax*nsrc`, broadcasts over the last axis of the reshaped `z`. The sign goes
on the **unconjugated** factor by convention; since `sgn` is real, either side is
mathematically identical.)

#### `gpu/gpu.py`

**Edits 8–9 — flag + signed sqrt (lines 113–115), identical in form to Edits 1–2:**
```python
# BEFORE
    coord_method_params = coord_method_params or {}
    coords = coord_method(
        flux=np.sqrt(0.5 * I_sky),

# AFTER
    coord_method_params = coord_method_params or {}
    negative_sky = bool(np.any(I_sky < 0))
    coords = coord_method(
        flux=np.sign(I_sky) * np.sqrt(0.5 * np.abs(I_sky)),
```
(The signed sqrt is computed on the host in NumPy; `CoordinateRotation.__init__` transfers it
to the GPU via `xp.asarray` unchanged — it stays real-typed, per the `iscomplexobj` check at
`core/coords.py:60–65`.)

**Edit 10 — stream loop (lines 193–229):**
```python
# BEFORE
            crdtop, Isqrt, nsrcs_up = coords.select_chunk(c, t)
            logdebug("crdtop", crdtop)
            logdebug("Isqrt", Isqrt)

            if nsrcs_up < 1:
                continue
            ...
            # compute vis = Z.Z^dagger
            matprod(z, c)

# AFTER
            crdtop, Isqrt, nsrcs_up = coords.select_chunk(c, t)
            logdebug("crdtop", crdtop)
            logdebug("Isqrt", Isqrt)

            if nsrcs_up < 1:
                continue

            sgn = cp.tile(cp.sign(Isqrt), nax) if negative_sky else None
            ...
            # compute vis = Z* diag(sign) Z^T
            matprod(z, c, sgn=sgn)
```
(`nax` is in scope — used at line 126; `cp` is already imported; `cp.sign`/`cp.tile` execute
on the chunk's active stream like the rest of the loop body.)

#### `gpu/matprod.py`

**Edit 11 — `GPUMatMul.compute` (lines 32–36):**
```python
# BEFORE
    def compute(self, z: cp.ndarray, out: cp.ndarray) -> cp.ndarray:
        """Perform the source-summing operation for a single time and chunk."""
        zdotz(z, out=out)
        cp.cuda.Device().synchronize()
        return out

# AFTER
    def compute(self, z, out, sgn=None):
        """Perform the source-summing operation for a single time and chunk."""
        if sgn is None:
            zdotz(z, out=out)                    # legacy path, bit-identical
        else:
            complex_matmul(z, z * sgn, out=out)  # a.conj() @ b.T with b = unsigned Z
        cp.cuda.Device().synchronize()
        return out
```
Contiguity asserts in `_cublas.complex_matmul` hold: first arg `z` is the original
C-contiguous buffer; `z * sgn` is a new C-contiguous array. `complex_matmul` is already
imported at line 7.

**Edit 12 — `GPUVectorDot.compute` (lines 81–89):**
```python
# BEFORE
    def compute(self, z: cp.ndarray, out: cp.ndarray) -> cp.ndarray:
        """Perform the source-summing operation for a single time and chunk."""
        z = z.reshape((self.nant, self.nfeed, -1))

        for i, (ai, aj) in enumerate(self.antpairs):
            complex_matmul(z[ai], z[aj], out=out[:, :, i])

# AFTER
    def compute(self, z, out, sgn=None):
        """Perform the source-summing operation for a single time and chunk."""
        z = z.reshape((self.nant, self.nfeed, -1))
        zs = z if sgn is None else z * sgn

        for i, (ai, aj) in enumerate(self.antpairs):
            complex_matmul(z[ai], zs[aj], out=out[:, :, i])
```

#### `wrapper.py`

**Edit 14 — docstring only.** `simulate_vis` (line 24) forwards `fluxes[:, i]` per frequency
(line 153) with no sign assumptions; add the same "may be negative" note to its `fluxes`
parameter docs.

### F.4 Explicitly unchanged files (and why)

- `core/getz.py` — builds `Z` from whatever flux it receives; the signed sqrt flows through
  `exptau *= sqrt_flux` (line 80) automatically. `exptau` is recomputed per chunk, so the
  in-place multiply cannot leak across chunks.
- `core/coords.py` — flux is only stored (65), sliced (112), horizon-masked (141), zero-padded
  (144); no arithmetic. The dtype logic (60–65) keeps a real signed sqrt real. The polarized
  coherency path (`flux.ndim == 4`) is unreachable from `simulate()` (1-D `I_sky` enforced by
  `_validate_inputs`).
- `core/__init__.py` (`_validate_inputs`) — only checks `I_sky.ndim == 1`; no sign check
  exists to remove.
- `core/tau.py`, `cpu/coords.py`, `gpu/coords.py`, `_utils.py` (chunk sizing / memory
  estimates), `MatProd.sum_chunks` — untouched by the flux path.
- `cpu/beams.py:46`, `gpu/beams.py:195` — the *other* square root (unpolarized **power
  beams** that interpolate negative in sidelobe nulls). Same failure pattern, **separate
  limitation**, deliberately out of scope for Fix 2.

### F.5 Downstream compatibility (audited in `myenv_validation_1`)

- **fftvis 1.0.1** — only references `matprod_method=""` as a CLI kwarg; no `MatProd`
  subclasses, no direct `compute()` calls.
- **hera_sim 4.3.3.dev8** (`visibilities/matvis.py:271`) — calls `cpu.simulate`/`gpu.simulate`
  passing `I_sky=sky_model.stokes[0, i].to("Jy").value` straight through; no positivity
  assumptions; `_reorder_vis` only copies values.
- **pyuvsim / pyuvdata** — receive final visibilities; no validation of auto positivity.
- `zdotz`/`complex_matmul` have **no callers outside `gpu/matprod.py`**; `matvis._test_utils`
  does not touch matprod. The `sgn=None` default keeps every existing call site
  source-compatible.
- **One behavioral change to flag:** with an indefinite sky, autocorrelations remain real but
  may be **negative** (correct physics — $V$ is no longer positive semi-definite). Any
  analysis code assuming positive autos should be checked.

### F.6 Verification plan

1. **Bit-exact regression:** all-non-negative sky ⇒ `negative_sky=False` ⇒ `sgn=None` ⇒ the
   exact legacy code path (`zdotz`, `z.conj().dot(z.T)`) runs; outputs must be bit-identical
   to unpatched v1.3.2.
2. **Ground truth:** mixed-sign toy sky vs. a direct RIME sum — the Appendix E demo cell above
   is exactly this test; extend it to call the patched `matvis.cpu.simulate`.
3. **Fix 1 cross-check:** `V_patched(I) == V(I⁺) − V(I⁻)` to float tolerance (two unpatched
   runs vs. one patched run).
4. **Hermiticity:** for conjugate antpairs, `V_pq == conj(V_qp)`.
5. **Polarized path:** run with `polarized=True` (nax=2, nfeed=2) and a mixed-sign sky to
   exercise the `np.tile(sgn, nax)` expansion.
6. **Upstream suite:** clone the matvis repo at `v1.3.2` (tests are not shipped in
   site-packages) and run its pytest suite against the patched tree.
7. **GPU:** repeat 1–5 on a GPU node if cupy is available in the env.
